# Null Catamenial Epilepsy Analysis Notebook

Populated from `outputs/random_start_full_v11_hormone_fix` outputs. Detected analysis mode: **full**.

- Participants: **100,000** (healthy ovulatory: 50000, heterogeneous menstruating-age: 50000)
- Primary window rows: **2,600,000**
- Study-level Monte Carlo rows: **1,380,000**
- Manifest files: **25**

## Cohort terminology

This notebook uses **heterogeneous menstruating-age** as the presentation label for the broader cohort key stored in the analysis files. In this null-simulation study it means an assumption-driven broader menstruating-age simulated cohort, not a disease-positive, clinically diagnosed, or demographically representative population. This cohort allows the hormone-cycle simulator's natural ovulatory and anovulatory behavior and its configured rates of cycle modifiers such as PCOS, peri-menarche, perimenopause, dysmenorrhea, and cycle irregularity when available. It is contrasted with the **healthy ovulatory** cohort, which is restricted to adult ovulatory cycling with those medical modifiers disabled where the simulator exposes those controls. In both cohorts, seizure and menstrual diaries are generated from separate deterministic random streams. HORMONE-CYCLE selects diary day 1 uniformly from the first generated cycle and then proceeds forward without wrapping. The seizure and menstrual diaries are aligned directly by calendar day without reordering, so any apparent catamenial epilepsy classification is a false positive under the null.

## Exact analysis plan followed

1. Simulate two cohorts separately and never pool results: healthy ovulatory and heterogeneous menstruating-age.
2. Full defined cohort sizes are `{'healthy ovulatory': 50000, 'heterogeneous menstruating-age': 50000}`; smoke mode uses `100` total participants.
3. For each participant, simulate an independent CHOCOLATES seizure diary and an independent hormone-cycle diary for `36` months in full mode.
4. Select diary day 1 uniformly from the first generated HORMONE-CYCLE cycle, continue forward without wrapping, and align the independently generated seizure and menstrual diaries directly by calendar day.
5. Label phases on the full diary before subsetting windows, using strict Herzog labels for primary analyses and a luteal-anchored fixed ovulatory window for sensitivity analyses.
6. Sample calendar windows, full 36-month windows, and complete-cycle windows exactly as configured.
7. Classify windows using exact Herzog 2004, windowed Herzog thresholds, C3-exclusion and pattern-only sensitivities, minimum-data rules, reproducibility rules, full-window stabilized/window-dispersion NB regression, and assumption-based historical definitions.
8. Summarize false positives and indeterminacy by cohort, phase mode, window, definition, seizure-burden stratum, participant-level status, pattern category, and study-level Monte Carlo benchmarks.
9. Save outputs as parquet/CSV, publication figures as PNG/PDF/SVG, a 1% daily audit sample, and a manifest.

### Recorded assumptions

- Definition D uses a participant-full-diary method-of-moments negative-binomial alpha recorded in d_alpha; Poisson robust fallback is recorded in d_reason when statsmodels NB fitting fails. Definition D_window_alpha re-estimates alpha from the analyzed window as a non-oracle sensitivity.
- HORMONE-CYCLE selected diary day 1 uniformly from the first generated cycle.
- Healthy ovulatory cohort used hormone_cycler build_patient_profile/render_cycle with ovulation_probability set to 1.0 because simulate_diary does not expose a public force-ovulation knob.
- Historical definitions H1-H4 are assumption-based operationalizations and are flagged in summary outputs.
- Study-level Monte Carlo samples each selected participant from a deterministic pool of precomputed random valid 3-month windows to avoid retaining all daily diaries in memory.
- The hormone simulator exposes medical-factor knobs but no natural prevalence sampler; heterogeneous menstruating-age medical factors were sampled from config.yaml rates.

## Reproducible function calls

These cells are the exact calls used to regenerate the analysis outputs. Run the smoke call for a quick end-to-end check; run the full call for the defined 100,000-participant analysis.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from paper1_null_ce.core.utils import load_config
from paper1_null_ce.core.simulate import run_pipeline

config = load_config(ROOT / "config_random_start_full.yaml")

# Quick validation run used while developing and reviewing the pipeline:
smoke_result = run_pipeline(config, mode="smoke")

# Prespecified full analysis. This is intentionally separate because it is large:
# full_result = run_pipeline(config, mode="full")

# During a long run, check ETA from another terminal:
# python3.11 scripts/check_paper1_progress.py --progress outputs/random_start_full_v11_hormone_fix/progress.json


[progress] initializing: 0/100 (0.0%), elapsed 0s - starting analysis run


[progress] participant_simulation: 0/100 (0.0%), elapsed 0s - starting cohort healthy_ovulatory


[progress] participant_simulation: 25/100 (25.0%), elapsed 3s, ETA 8s, 9.10 participants/s - healthy_ovulatory participants 1-25 completed in 3s


[progress] participant_simulation: 50/100 (50.0%), elapsed 5s, ETA 5s, 9.76 participants/s - healthy_ovulatory participants 26-50 completed in 2s


[progress] participant_simulation: 50/100 (50.0%), elapsed 5s, ETA 5s, 9.75 participants/s - starting cohort population


...

[progress] summary_tables: 100/100 (100.0%), elapsed 14s, 7.07 participants/s - summarizing false-positive and indeterminate rates


[progress] writing_outputs: 100/100 (100.0%), elapsed 20s, 4.89 participants/s - writing parquet/csv outputs


[progress] figures: 100/100 (100.0%), elapsed 21s, 4.79 participants/s - writing publication figures


[progress] manifest: 100/100 (100.0%), elapsed 22s, 4.50 participants/s - writing manifest


[progress] complete: 100/100 (100.0%), elapsed 22s, 4.50 participants/s - analysis complete in 22s


## Load the current populated outputs

The remaining notebook cells read the existing output artifacts. This keeps figure and table rendering fast and reproducible after either a smoke run or a full run.

In [2]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = ROOT / "outputs/random_start_full_v11_hormone_fix"

participant_summary = pd.read_parquet(OUTPUT_DIR / "participant_summary.parquet")
window_results = pd.read_parquet(OUTPUT_DIR / "window_results.parquet")
study_path = OUTPUT_DIR / "study_level_3month.parquet"
if not study_path.exists():
    study_path = OUTPUT_DIR / "study_level_3month_n30.parquet"
study_level = pd.read_parquet(study_path)
summary_tables = pd.read_csv(OUTPUT_DIR / "summary_tables.csv")
manifest = json.loads((OUTPUT_DIR / "manifest.json").read_text())

participant_summary.shape, window_results.shape, study_level.shape, summary_tables.shape


/Users/runner/work/crossbow/crossbow/arrow/cpp/src/arrow/util/cpu_info.cc:242: IOError: sysctlbyname failed for 'hw.l1dcachesize'. Detail: [errno 1] Operation not permitted
/Users/runner/work/crossbow/crossbow/arrow/cpp/src/arrow/util/cpu_info.cc:242: IOError: sysctlbyname failed for 'hw.l2cachesize'. Detail: [errno 1] Operation not permitted
/Users/runner/work/crossbow/crossbow/arrow/cpp/src/arrow/util/cpu_info.cc:242: IOError: sysctlbyname failed for 'hw.l3cachesize'. Detail: [errno 1] Operation not permitted
/Users/runner/work/crossbow/crossbow/arrow/cpp/src/arrow/util/cpu_info.cc:242: IOError: sysctlbyname failed for 'hw.optional.neon'. Detail: [errno 1] Operation not permitted


((100000, 27), (2600000, 109), (1380000, 9), (19701, 24))

## Table 1. Cohort and diary summary

**Why this table is included.** This table verifies that both defined cohorts are represented separately, that cycle summaries are available, and that seizure-burden metrics were carried through from the seizure simulator. It is the first QC table because every downstream apparent-classification estimate depends on the cohort construction and diary burden.

**Code to call.**

In [3]:
cohort_summary = (
    participant_summary
    .groupby("cohort")
    .agg(
        participants=("participant_id", "nunique"),
        age_mean=("age", "mean"),
        age_sd=("age", "std"),
        mean_cycle_length=("mean_cycle_length", "mean"),
        sd_cycle_length=("sd_cycle_length", "mean"),
        ovulatory_fraction=("ovulatory_fraction", "mean"),
        seizure_days_per_month=("seizure_days_per_month", "mean"),
        seizures_per_month=("seizures_per_month", "mean"),
    )
    .reset_index()
)
cohort_summary


,cohort,participants,age_mean,age_sd,mean_cycle_length,sd_cycle_length,ovulatory_fraction,seizure_days_per_month,seizures_per_month
0,healthy_ovulatory,50000,31.519764,7.801985,29.150240,3.395223,1.000000,2.458204,6.831809
1,population,50000,33.968544,12.138328,30.920656,4.740557,0.789516,2.454090,6.796666


| Cohort                         | Participants | Mean age, years | Age SD, years | Mean cycle length, days | Mean cycle-length SD, days | Ovulatory cycles | Seizure days per month | Seizures per month |
| ------------------------------ | ------------ | --------------- | ------------- | ----------------------- | -------------------------- | ---------------- | ---------------------- | ------------------ |
| healthy ovulatory              | 50,000       | 31.5            | 7.8           | 29.15                   | 3.40                       | 100.0%           | 2.46                   | 6.83               |
| heterogeneous menstruating-age | 50,000       | 34.0            | 12.1          | 30.92                   | 4.74                       | 79.0%            | 2.45                   | 6.80               |

**Table 1 caption.** Cohort-level participant and diary summaries for the full simulation. Percentages use a 0-100% scale; seizure rates are monthly averages over the 36-month diary.

## Table 2. Primary full-window false-positive rates

**Why this table is included.** This table is the primary result summary for person-window false-positive rates under the null. It uses the full diary window and reports classifiable denominators, positives, Wilson 95% intervals, and indeterminate rates separately by cohort and definition.

**Code to call.**

In [4]:
primary_full = summary_tables[
    (summary_tables.table_type == "window_false_positive")
    & (summary_tables.phase_mode == "strict_herzog")
    & (summary_tables.subset == "all")
    & (summary_tables.window_type == "full")
    & (summary_tables.definition.isin([
        "A_windowed_any", "A_windowed_C1_or_C2",
        "B_minimum_data_any", "B_minimum_data_C1_or_C2",
        "C_reproducibility_any", "D_nb_regression_C1_or_C2"
    ]))
].copy()
primary_full


,table_type,subset,cohort,window_type,window_value,definition,phase_mode,assumption_based_historical,n_windows,n_classifiable,...,indeterminate_rate,positive_rate_all_attempted,unstable_denominator,interpretation_note,n_participants,pattern_category,indeterminate_reason,n_indeterminate,p_prevalence_ge_39_1,p_prevalence_ge_44_2
77,window_false_positive,all,healthy_ovulatory,full,full_diary,A_windowed_any,strict_herzog,False,50000,49605.0,...,0.00790,0.11592,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
103,window_false_positive,all,population,full,full_diary,A_windowed_any,strict_herzog,False,50000,49587.0,...,0.00826,0.35858,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
181,window_false_positive,all,healthy_ovulatory,full,full_diary,A_windowed_C1_or_C2,strict_herzog,False,50000,49605.0,...,0.00790,0.11592,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
207,window_false_positive,all,population,full,full_diary,A_windowed_C1_or_C2,strict_herzog,False,50000,49587.0,...,0.00826,0.11574,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
389,window_false_positive,all,healthy_ovulatory,full,full_diary,B_minimum_data_any,strict_herzog,False,50000,48359.0,...,0.03282,0.10308,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
415,window_false_positive,all,population,full,full_diary,B_minimum_data_any,strict_herzog,False,50000,48333.0,...,0.03334,0.34362,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
493,window_false_positive,all,healthy_ovulatory,full,full_diary,B_minimum_data_C1_or_C2,strict_herzog,False,50000,48359.0,...,0.03282,0.10308,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
519,window_false_positive,all,population,full,full_diary,B_minimum_data_C1_or_C2,strict_herzog,False,50000,48333.0,...,0.03334,0.10294,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
545,window_false_positive,all,healthy_ovulatory,full,full_diary,C_reproducibility_any,strict_herzog,False,50000,6318.0,...,0.87364,0.00000,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
571,window_false_positive,all,population,full,full_diary,C_reproducibility_any,strict_herzog,False,50000,13341.0,...,0.73318,0.03114,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN


| Cohort                         | CE definition                           | Windows analyzed | Classifiable windows | False-positive windows | False-positive rate (95% CI) | Indeterminate windows |
| ------------------------------ | --------------------------------------- | ---------------- | -------------------- | ---------------------- | ---------------------------- | --------------------- |
| healthy ovulatory              | Windowed Herzog C1/C2 union             | 50,000           | 49,605               | 5,796                  | 11.7% (11.4, 12.0)           | 0.8%                  |
| healthy ovulatory              | Windowed Herzog thresholds              | 50,000           | 49,605               | 5,796                  | 11.7% (11.4, 12.0)           | 0.8%                  |
| healthy ovulatory              | Windowed Herzog C1/C2 with minimum data | 50,000           | 48,359               | 5,154                  | 10.7% (10.4, 10.9)           | 3.3%                  |
| healthy ovulatory              | Windowed Herzog with minimum data       | 50,000           | 48,359               | 5,154                  | 10.7% (10.4, 10.9)           | 3.3%                  |
| healthy ovulatory              | Cycle reproducibility, 6-cycle rule     | 50,000           | 6,318                | 0                      | 0.0% (0.0, 0.1)              | 87.4%                 |
| healthy ovulatory              | Negative-binomial regression C1/C2      | 50,000           | 48,359               | 2,125                  | 4.4% (4.2, 4.6)              | 3.3%                  |
| heterogeneous menstruating-age | Windowed Herzog C1/C2 union             | 50,000           | 49,587               | 5,787                  | 11.7% (11.4, 12.0)           | 0.8%                  |
| heterogeneous menstruating-age | Windowed Herzog thresholds              | 50,000           | 49,587               | 17,929                 | 36.2% (35.7, 36.6)           | 0.8%                  |
| heterogeneous menstruating-age | Windowed Herzog C1/C2 with minimum data | 50,000           | 48,333               | 5,147                  | 10.6% (10.4, 10.9)           | 3.3%                  |
| heterogeneous menstruating-age | Windowed Herzog with minimum data       | 50,000           | 48,333               | 17,181                 | 35.5% (35.1, 36.0)           | 3.3%                  |
| heterogeneous menstruating-age | Cycle reproducibility, 6-cycle rule     | 50,000           | 13,341               | 1,557                  | 11.7% (11.1, 12.2)           | 73.3%                 |
| heterogeneous menstruating-age | Negative-binomial regression C1/C2      | 50,000           | 48,333               | 2,035                  | 4.2% (4.0, 4.4)              | 3.3%                  |

**Table 2 caption.** Primary full-diary false-positive rates under the null. The denominator for the false-positive rate is the number of classifiable participant windows, and the confidence interval is Wilson 95%.

## Table 3. Window-length sensitivity for core definitions

**Why this table is included.** This table shows why diary length matters. Short calendar windows can be classifiable for simple windowed ratios but not for minimum-data, reproducibility, or exact three-cycle rules; the indeterminate column quantifies that tradeoff.

**Code to call.**

In [5]:
window_sensitivity = summary_tables[
    (summary_tables.table_type == "window_false_positive")
    & (summary_tables.phase_mode == "strict_herzog")
    & (summary_tables.subset == "all")
    & (summary_tables.definition.isin([
        "A_exact_any", "A_windowed_any", "A_windowed_C1_or_C2",
        "B_minimum_data_C1_or_C2", "C_reproducibility_C1_or_C2",
        "D_nb_regression_C1_or_C2"
    ]))
].copy()
window_sensitivity


,table_type,subset,cohort,window_type,window_value,definition,phase_mode,assumption_based_historical,n_windows,n_classifiable,...,indeterminate_rate,positive_rate_all_attempted,unstable_denominator,interpretation_note,n_participants,pattern_category,indeterminate_reason,n_indeterminate,p_prevalence_ge_39_1,p_prevalence_ge_44_2
13,window_false_positive,all,healthy_ovulatory,calendar,1,A_exact_any,strict_herzog,False,50000,0.0,...,1.00000,0.0000,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14,window_false_positive,all,healthy_ovulatory,calendar,3,A_exact_any,strict_herzog,False,50000,0.0,...,1.00000,0.0000,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15,window_false_positive,all,healthy_ovulatory,calendar,4,A_exact_any,strict_herzog,False,50000,0.0,...,1.00000,0.0000,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16,window_false_positive,all,healthy_ovulatory,calendar,6,A_exact_any,strict_herzog,False,50000,0.0,...,1.00000,0.0000,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,window_false_positive,all,healthy_ovulatory,calendar,9,A_exact_any,strict_herzog,False,50000,0.0,...,1.00000,0.0000,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
827,window_false_positive,all,population,calendar,36,D_nb_regression_C1_or_C2,strict_herzog,False,50000,0.0,...,1.00000,0.0000,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
828,window_false_positive,all,population,cycle,3,D_nb_regression_C1_or_C2,strict_herzog,False,50000,0.0,...,1.00000,0.0000,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
829,window_false_positive,all,population,cycle,6,D_nb_regression_C1_or_C2,strict_herzog,False,50000,0.0,...,1.00000,0.0000,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
830,window_false_positive,all,population,cycle,12,D_nb_regression_C1_or_C2,strict_herzog,False,50000,0.0,...,1.00000,0.0000,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN


| Cohort                         | Observation window  | CE definition                             | Classifiable windows | False-positive windows | False-positive rate | Indeterminate windows |
| ------------------------------ | ------------------- | ----------------------------------------- | -------------------- | ---------------------- | ------------------- | --------------------- |
| healthy ovulatory              | 1 month             | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 1 month             | Windowed Herzog C1/C2 union               | 38,214               | 18,163                 | 47.5%               | 23.6%                 |
| healthy ovulatory              | 1 month             | Windowed Herzog thresholds                | 38,214               | 18,163                 | 47.5%               | 23.6%                 |
| healthy ovulatory              | 1 month             | Windowed Herzog C1/C2 with minimum data   | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 1 month             | Cycle reproducibility C1/C2, 6-cycle rule | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 1 month             | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 12 cycles           | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 12 cycles           | Windowed Herzog C1/C2 union               | 48,676               | 12,362                 | 25.4%               | 2.6%                  |
| healthy ovulatory              | 12 cycles           | Windowed Herzog thresholds                | 48,676               | 12,362                 | 25.4%               | 2.6%                  |
| healthy ovulatory              | 12 cycles           | Windowed Herzog C1/C2 with minimum data   | 45,106               | 10,508                 | 23.3%               | 9.8%                  |
| healthy ovulatory              | 12 cycles           | Cycle reproducibility C1/C2, 6-cycle rule | 15,651               | 312                    | 2.0%                | 68.7%                 |
| healthy ovulatory              | 12 cycles           | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 12 months           | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 12 months           | Windowed Herzog C1/C2 union               | 48,731               | 12,442                 | 25.5%               | 2.5%                  |
| healthy ovulatory              | 12 months           | Windowed Herzog thresholds                | 48,731               | 12,442                 | 25.5%               | 2.5%                  |
| healthy ovulatory              | 12 months           | Windowed Herzog C1/C2 with minimum data   | 45,268               | 10,654                 | 23.5%               | 9.5%                  |
| healthy ovulatory              | 12 months           | Cycle reproducibility C1/C2, 6-cycle rule | 16,050               | 250                    | 1.6%                | 67.9%                 |
| healthy ovulatory              | 12 months           | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 18 months           | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 18 months           | Windowed Herzog C1/C2 union               | 49,159               | 9,763                  | 19.9%               | 1.7%                  |
| healthy ovulatory              | 18 months           | Windowed Herzog thresholds                | 49,159               | 9,763                  | 19.9%               | 1.7%                  |
| healthy ovulatory              | 18 months           | Windowed Herzog C1/C2 with minimum data   | 46,723               | 8,532                  | 18.3%               | 6.6%                  |
| healthy ovulatory              | 18 months           | Cycle reproducibility C1/C2, 6-cycle rule | 12,201               | 25                     | 0.2%                | 75.6%                 |
| healthy ovulatory              | 18 months           | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 24 months           | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 24 months           | Windowed Herzog C1/C2 union               | 49,381               | 7,869                  | 15.9%               | 1.2%                  |
| healthy ovulatory              | 24 months           | Windowed Herzog thresholds                | 49,381               | 7,869                  | 15.9%               | 1.2%                  |
| healthy ovulatory              | 24 months           | Windowed Herzog C1/C2 with minimum data   | 47,535               | 6,949                  | 14.6%               | 4.9%                  |
| healthy ovulatory              | 24 months           | Cycle reproducibility C1/C2, 6-cycle rule | 9,522                | 3                      | 0.0%                | 81.0%                 |
| healthy ovulatory              | 24 months           | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 3 cycles            | Exact Herzog 2004, any CE pattern         | 23,200               | 11,555                 | 49.8%               | 53.6%                 |
| healthy ovulatory              | 3 cycles            | Windowed Herzog C1/C2 union               | 45,133               | 18,538                 | 41.1%               | 9.7%                  |
| healthy ovulatory              | 3 cycles            | Windowed Herzog thresholds                | 45,133               | 18,538                 | 41.1%               | 9.7%                  |
| healthy ovulatory              | 3 cycles            | Windowed Herzog C1/C2 with minimum data   | 16                   | 3                      | 18.8%               | 100.0%                |
| healthy ovulatory              | 3 cycles            | Cycle reproducibility C1/C2, 6-cycle rule | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 3 cycles            | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 3 months            | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 3 months            | Windowed Herzog C1/C2 union               | 45,321               | 18,758                 | 41.4%               | 9.4%                  |
| healthy ovulatory              | 3 months            | Windowed Herzog thresholds                | 45,321               | 18,758                 | 41.4%               | 9.4%                  |
| healthy ovulatory              | 3 months            | Windowed Herzog C1/C2 with minimum data   | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 3 months            | Cycle reproducibility C1/C2, 6-cycle rule | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 3 months            | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 36 months           | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 36 months           | Windowed Herzog C1/C2 union               | 49,605               | 5,796                  | 11.7%               | 0.8%                  |
| healthy ovulatory              | 36 months           | Windowed Herzog thresholds                | 49,605               | 5,796                  | 11.7%               | 0.8%                  |
| healthy ovulatory              | 36 months           | Windowed Herzog C1/C2 with minimum data   | 48,359               | 5,154                  | 10.7%               | 3.3%                  |
| healthy ovulatory              | 36 months           | Cycle reproducibility C1/C2, 6-cycle rule | 6,318                | 0                      | 0.0%                | 87.4%                 |
| healthy ovulatory              | 36 months           | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 36-month full diary | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 36-month full diary | Windowed Herzog C1/C2 union               | 49,605               | 5,796                  | 11.7%               | 0.8%                  |
| healthy ovulatory              | 36-month full diary | Windowed Herzog thresholds                | 49,605               | 5,796                  | 11.7%               | 0.8%                  |
| healthy ovulatory              | 36-month full diary | Windowed Herzog C1/C2 with minimum data   | 48,359               | 5,154                  | 10.7%               | 3.3%                  |
| healthy ovulatory              | 36-month full diary | Cycle reproducibility C1/C2, 6-cycle rule | 6,318                | 0                      | 0.0%                | 87.4%                 |
| healthy ovulatory              | 36-month full diary | Negative-binomial regression C1/C2        | 48,359               | 2,125                  | 4.4%                | 3.3%                  |
| healthy ovulatory              | 4 months            | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 4 months            | Windowed Herzog C1/C2 union               | 46,302               | 18,197                 | 39.3%               | 7.4%                  |
| healthy ovulatory              | 4 months            | Windowed Herzog thresholds                | 46,302               | 18,197                 | 39.3%               | 7.4%                  |
| healthy ovulatory              | 4 months            | Windowed Herzog C1/C2 with minimum data   | 37,627               | 13,737                 | 36.5%               | 24.7%                 |
| healthy ovulatory              | 4 months            | Cycle reproducibility C1/C2, 6-cycle rule | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 4 months            | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 6 cycles            | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 6 cycles            | Windowed Herzog C1/C2 union               | 47,425               | 16,243                 | 34.2%               | 5.1%                  |
| healthy ovulatory              | 6 cycles            | Windowed Herzog thresholds                | 47,425               | 16,243                 | 34.2%               | 5.1%                  |
| healthy ovulatory              | 6 cycles            | Windowed Herzog C1/C2 with minimum data   | 40,919               | 12,954                 | 31.7%               | 18.2%                 |
| healthy ovulatory              | 6 cycles            | Cycle reproducibility C1/C2, 6-cycle rule | 22,716               | 2,734                  | 12.0%               | 54.6%                 |
| healthy ovulatory              | 6 cycles            | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 6 months            | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 6 months            | Windowed Herzog C1/C2 union               | 47,533               | 16,223                 | 34.1%               | 4.9%                  |
| healthy ovulatory              | 6 months            | Windowed Herzog thresholds                | 47,533               | 16,223                 | 34.1%               | 4.9%                  |
| healthy ovulatory              | 6 months            | Windowed Herzog C1/C2 with minimum data   | 41,140               | 12,950                 | 31.5%               | 17.7%                 |
| healthy ovulatory              | 6 months            | Cycle reproducibility C1/C2, 6-cycle rule | 6,808                | 733                    | 10.8%               | 86.4%                 |
| healthy ovulatory              | 6 months            | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 9 months            | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 9 months            | Windowed Herzog C1/C2 union               | 48,331               | 14,277                 | 29.5%               | 3.3%                  |
| healthy ovulatory              | 9 months            | Windowed Herzog thresholds                | 48,331               | 14,277                 | 29.5%               | 3.3%                  |
| healthy ovulatory              | 9 months            | Windowed Herzog C1/C2 with minimum data   | 43,805               | 11,978                 | 27.3%               | 12.4%                 |
| healthy ovulatory              | 9 months            | Cycle reproducibility C1/C2, 6-cycle rule | 19,132               | 685                    | 3.6%                | 61.7%                 |
| healthy ovulatory              | 9 months            | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 1 month             | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 1 month             | Windowed Herzog C1/C2 union               | 38,138               | 18,435                 | 48.3%               | 23.7%                 |
| heterogeneous menstruating-age | 1 month             | Windowed Herzog thresholds                | 38,138               | 19,894                 | 52.2%               | 23.7%                 |
| heterogeneous menstruating-age | 1 month             | Windowed Herzog C1/C2 with minimum data   | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 1 month             | Cycle reproducibility C1/C2, 6-cycle rule | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 1 month             | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 12 cycles           | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 12 cycles           | Windowed Herzog C1/C2 union               | 48,750               | 12,214                 | 25.1%               | 2.5%                  |
| heterogeneous menstruating-age | 12 cycles           | Windowed Herzog thresholds                | 48,750               | 20,226                 | 41.5%               | 2.5%                  |
| heterogeneous menstruating-age | 12 cycles           | Windowed Herzog C1/C2 with minimum data   | 45,366               | 10,461                 | 23.1%               | 9.3%                  |
| heterogeneous menstruating-age | 12 cycles           | Cycle reproducibility C1/C2, 6-cycle rule | 18,653               | 158                    | 0.8%                | 62.7%                 |
| heterogeneous menstruating-age | 12 cycles           | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 12 months           | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 12 months           | Windowed Herzog C1/C2 union               | 48,719               | 12,472                 | 25.6%               | 2.6%                  |
| heterogeneous menstruating-age | 12 months           | Windowed Herzog thresholds                | 48,719               | 20,656                 | 42.4%               | 2.6%                  |
| heterogeneous menstruating-age | 12 months           | Windowed Herzog C1/C2 with minimum data   | 45,278               | 10,692                 | 23.6%               | 9.4%                  |
| heterogeneous menstruating-age | 12 months           | Cycle reproducibility C1/C2, 6-cycle rule | 19,625               | 110                    | 0.6%                | 60.7%                 |
| heterogeneous menstruating-age | 12 months           | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 18 months           | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 18 months           | Windowed Herzog C1/C2 union               | 49,139               | 9,748                  | 19.8%               | 1.7%                  |
| heterogeneous menstruating-age | 18 months           | Windowed Herzog thresholds                | 49,139               | 19,462                 | 39.6%               | 1.7%                  |
| heterogeneous menstruating-age | 18 months           | Windowed Herzog C1/C2 with minimum data   | 46,757               | 8,588                  | 18.4%               | 6.5%                  |
| heterogeneous menstruating-age | 18 months           | Cycle reproducibility C1/C2, 6-cycle rule | 15,578               | 21                     | 0.1%                | 68.8%                 |
| heterogeneous menstruating-age | 18 months           | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 24 months           | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 24 months           | Windowed Herzog C1/C2 union               | 49,373               | 7,942                  | 16.1%               | 1.3%                  |
| heterogeneous menstruating-age | 24 months           | Windowed Herzog thresholds                | 49,373               | 18,621                 | 37.7%               | 1.3%                  |
| heterogeneous menstruating-age | 24 months           | Windowed Herzog C1/C2 with minimum data   | 47,514               | 7,026                  | 14.8%               | 5.0%                  |
| heterogeneous menstruating-age | 24 months           | Cycle reproducibility C1/C2, 6-cycle rule | 12,757               | 2                      | 0.0%                | 74.5%                 |
| heterogeneous menstruating-age | 24 months           | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 3 cycles            | Exact Herzog 2004, any CE pattern         | 16,689               | 8,589                  | 51.5%               | 66.6%                 |
| heterogeneous menstruating-age | 3 cycles            | Windowed Herzog C1/C2 union               | 45,350               | 18,575                 | 41.0%               | 9.3%                  |
| heterogeneous menstruating-age | 3 cycles            | Windowed Herzog thresholds                | 45,350               | 22,781                 | 50.2%               | 9.3%                  |
| heterogeneous menstruating-age | 3 cycles            | Windowed Herzog C1/C2 with minimum data   | 1,991                | 734                    | 36.9%               | 96.0%                 |
| heterogeneous menstruating-age | 3 cycles            | Cycle reproducibility C1/C2, 6-cycle rule | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 3 cycles            | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 3 months            | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 3 months            | Windowed Herzog C1/C2 union               | 45,198               | 18,828                 | 41.7%               | 9.6%                  |
| heterogeneous menstruating-age | 3 months            | Windowed Herzog thresholds                | 45,198               | 22,901                 | 50.7%               | 9.6%                  |
| heterogeneous menstruating-age | 3 months            | Windowed Herzog C1/C2 with minimum data   | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 3 months            | Cycle reproducibility C1/C2, 6-cycle rule | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 3 months            | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 36 months           | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 36 months           | Windowed Herzog C1/C2 union               | 49,587               | 5,787                  | 11.7%               | 0.8%                  |
| heterogeneous menstruating-age | 36 months           | Windowed Herzog thresholds                | 49,587               | 17,929                 | 36.2%               | 0.8%                  |
| heterogeneous menstruating-age | 36 months           | Windowed Herzog C1/C2 with minimum data   | 48,333               | 5,147                  | 10.6%               | 3.3%                  |
| heterogeneous menstruating-age | 36 months           | Cycle reproducibility C1/C2, 6-cycle rule | 9,227                | 1                      | 0.0%                | 81.5%                 |
| heterogeneous menstruating-age | 36 months           | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 36-month full diary | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 36-month full diary | Windowed Herzog C1/C2 union               | 49,587               | 5,787                  | 11.7%               | 0.8%                  |
| heterogeneous menstruating-age | 36-month full diary | Windowed Herzog thresholds                | 49,587               | 17,929                 | 36.2%               | 0.8%                  |
| heterogeneous menstruating-age | 36-month full diary | Windowed Herzog C1/C2 with minimum data   | 48,333               | 5,147                  | 10.6%               | 3.3%                  |
| heterogeneous menstruating-age | 36-month full diary | Cycle reproducibility C1/C2, 6-cycle rule | 9,227                | 1                      | 0.0%                | 81.5%                 |
| heterogeneous menstruating-age | 36-month full diary | Negative-binomial regression C1/C2        | 48,333               | 2,035                  | 4.2%                | 3.3%                  |
| heterogeneous menstruating-age | 4 months            | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 4 months            | Windowed Herzog C1/C2 union               | 46,301               | 18,139                 | 39.2%               | 7.4%                  |
| heterogeneous menstruating-age | 4 months            | Windowed Herzog thresholds                | 46,301               | 23,022                 | 49.7%               | 7.4%                  |
| heterogeneous menstruating-age | 4 months            | Windowed Herzog C1/C2 with minimum data   | 37,366               | 13,517                 | 36.2%               | 25.3%                 |
| heterogeneous menstruating-age | 4 months            | Cycle reproducibility C1/C2, 6-cycle rule | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 4 months            | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 6 cycles            | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 6 cycles            | Windowed Herzog C1/C2 union               | 47,597               | 16,096                 | 33.8%               | 4.8%                  |
| heterogeneous menstruating-age | 6 cycles            | Windowed Herzog thresholds                | 47,597               | 22,048                 | 46.3%               | 4.8%                  |
| heterogeneous menstruating-age | 6 cycles            | Windowed Herzog C1/C2 with minimum data   | 41,299               | 12,744                 | 30.9%               | 17.4%                 |
| heterogeneous menstruating-age | 6 cycles            | Cycle reproducibility C1/C2, 6-cycle rule | 25,709               | 1,631                  | 6.3%                | 48.6%                 |
| heterogeneous menstruating-age | 6 cycles            | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 6 months            | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 6 months            | Windowed Herzog C1/C2 union               | 47,453               | 16,386                 | 34.5%               | 5.1%                  |
| heterogeneous menstruating-age | 6 months            | Windowed Herzog thresholds                | 47,453               | 22,425                 | 47.3%               | 5.1%                  |
| heterogeneous menstruating-age | 6 months            | Windowed Herzog C1/C2 with minimum data   | 41,163               | 13,114                 | 31.9%               | 17.7%                 |
| heterogeneous menstruating-age | 6 months            | Cycle reproducibility C1/C2, 6-cycle rule | 5,161                | 395                    | 7.7%                | 89.7%                 |
| heterogeneous menstruating-age | 6 months            | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 9 months            | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 9 months            | Windowed Herzog C1/C2 union               | 48,306               | 14,165                 | 29.3%               | 3.4%                  |
| heterogeneous menstruating-age | 9 months            | Windowed Herzog thresholds                | 48,306               | 21,349                 | 44.2%               | 3.4%                  |
| heterogeneous menstruating-age | 9 months            | Windowed Herzog C1/C2 with minimum data   | 43,903               | 11,946                 | 27.2%               | 12.2%                 |
| heterogeneous menstruating-age | 9 months            | Cycle reproducibility C1/C2, 6-cycle rule | 21,686               | 375                    | 1.7%                | 56.6%                 |
| heterogeneous menstruating-age | 9 months            | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |

**Table 3 caption.** False-positive and indeterminate rates for every prespecified observation window and core definition. Exact Herzog 2004 is expected to be classifiable only for 3-complete-cycle windows.

## Table 4. Strict Herzog versus luteal-anchored ovulatory sensitivity

**Why this table is included.** This table addresses whether false-positive rates depend on the strict Herzog periovulatory window expanding with cycle length. Strict Herzog remains primary for historical comparability; the luteal-anchored mode fixes the ovulatory window at four pre-luteal days.

**Code to call.**

In [6]:
phase_mode_sensitivity = summary_tables[
    (summary_tables.table_type == "window_false_positive")
    & (summary_tables.subset == "all")
    & (
        (summary_tables.window_type == "full")
        | ((summary_tables.window_type == "calendar") & (summary_tables.window_value.astype(str) == "3"))
    )
    & (summary_tables.definition.isin(["A_windowed_any", "A_windowed_C1_or_C2", "A_windowed_C3_only"]))
].copy()
phase_mode_sensitivity


,table_type,subset,cohort,window_type,window_value,definition,phase_mode,assumption_based_historical,n_windows,n_classifiable,...,indeterminate_rate,positive_rate_all_attempted,unstable_denominator,interpretation_note,n_participants,pattern_category,indeterminate_reason,n_indeterminate,p_prevalence_ge_39_1,p_prevalence_ge_44_2
53,window_false_positive,all,healthy_ovulatory,calendar,3,A_windowed_any,luteal_anchored_ovulatory,False,50000,45321.0,...,0.09358,0.36182,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
64,window_false_positive,all,healthy_ovulatory,full,full_diary,A_windowed_any,luteal_anchored_ovulatory,False,50000,49605.0,...,0.00790,0.12280,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
66,window_false_positive,all,healthy_ovulatory,calendar,3,A_windowed_any,strict_herzog,False,50000,45321.0,...,0.09358,0.37516,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
77,window_false_positive,all,healthy_ovulatory,full,full_diary,A_windowed_any,strict_herzog,False,50000,49605.0,...,0.00790,0.11592,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
79,window_false_positive,all,population,calendar,3,A_windowed_any,luteal_anchored_ovulatory,False,50000,45198.0,...,0.09604,0.41910,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
90,window_false_positive,all,population,full,full_diary,A_windowed_any,luteal_anchored_ovulatory,False,50000,49587.0,...,0.00826,0.29514,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
92,window_false_positive,all,population,calendar,3,A_windowed_any,strict_herzog,False,50000,45198.0,...,0.09604,0.45802,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
103,window_false_positive,all,population,full,full_diary,A_windowed_any,strict_herzog,False,50000,49587.0,...,0.00826,0.35858,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
157,window_false_positive,all,healthy_ovulatory,calendar,3,A_windowed_C1_or_C2,luteal_anchored_ovulatory,False,50000,45321.0,...,0.09358,0.36182,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
168,window_false_positive,all,healthy_ovulatory,full,full_diary,A_windowed_C1_or_C2,luteal_anchored_ovulatory,False,50000,49605.0,...,0.00790,0.12280,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN


| Cohort                         | Phase labeling            | Observation window  | CE definition               | Classifiable windows | False-positive windows | False-positive rate (95% CI) | Indeterminate windows |
| ------------------------------ | ------------------------- | ------------------- | --------------------------- | -------------------- | ---------------------- | ---------------------------- | --------------------- |
| healthy ovulatory              | Luteal-anchored ovulatory | 3 months            | Windowed Herzog C1/C2 union | 45,321               | 18,091                 | 39.9% (39.5, 40.4)           | 9.4%                  |
| healthy ovulatory              | Strict Herzog             | 3 months            | Windowed Herzog C1/C2 union | 45,321               | 18,758                 | 41.4% (40.9, 41.8)           | 9.4%                  |
| healthy ovulatory              | Luteal-anchored ovulatory | 3 months            | Windowed Herzog C3 only     | 0                    | 0                      | NA                           | 100.0%                |
| healthy ovulatory              | Strict Herzog             | 3 months            | Windowed Herzog C3 only     | 0                    | 0                      | NA                           | 100.0%                |
| healthy ovulatory              | Luteal-anchored ovulatory | 3 months            | Windowed Herzog thresholds  | 45,321               | 18,091                 | 39.9% (39.5, 40.4)           | 9.4%                  |
| healthy ovulatory              | Strict Herzog             | 3 months            | Windowed Herzog thresholds  | 45,321               | 18,758                 | 41.4% (40.9, 41.8)           | 9.4%                  |
| healthy ovulatory              | Luteal-anchored ovulatory | 36-month full diary | Windowed Herzog C1/C2 union | 49,605               | 6,140                  | 12.4% (12.1, 12.7)           | 0.8%                  |
| healthy ovulatory              | Strict Herzog             | 36-month full diary | Windowed Herzog C1/C2 union | 49,605               | 5,796                  | 11.7% (11.4, 12.0)           | 0.8%                  |
| healthy ovulatory              | Luteal-anchored ovulatory | 36-month full diary | Windowed Herzog C3 only     | 0                    | 0                      | NA                           | 100.0%                |
| healthy ovulatory              | Strict Herzog             | 36-month full diary | Windowed Herzog C3 only     | 0                    | 0                      | NA                           | 100.0%                |
| healthy ovulatory              | Luteal-anchored ovulatory | 36-month full diary | Windowed Herzog thresholds  | 49,605               | 6,140                  | 12.4% (12.1, 12.7)           | 0.8%                  |
| healthy ovulatory              | Strict Herzog             | 36-month full diary | Windowed Herzog thresholds  | 49,605               | 5,796                  | 11.7% (11.4, 12.0)           | 0.8%                  |
| heterogeneous menstruating-age | Luteal-anchored ovulatory | 3 months            | Windowed Herzog C1/C2 union | 45,198               | 17,975                 | 39.8% (39.3, 40.2)           | 9.6%                  |
| heterogeneous menstruating-age | Strict Herzog             | 3 months            | Windowed Herzog C1/C2 union | 45,198               | 18,828                 | 41.7% (41.2, 42.1)           | 9.6%                  |
| heterogeneous menstruating-age | Luteal-anchored ovulatory | 3 months            | Windowed Herzog C3 only     | 16,916               | 6,751                  | 39.9% (39.2, 40.6)           | 66.2%                 |
| heterogeneous menstruating-age | Strict Herzog             | 3 months            | Windowed Herzog C3 only     | 16,491               | 8,904                  | 54.0% (53.2, 54.8)           | 67.0%                 |
| heterogeneous menstruating-age | Luteal-anchored ovulatory | 3 months            | Windowed Herzog thresholds  | 45,198               | 20,955                 | 46.4% (45.9, 46.8)           | 9.6%                  |
| heterogeneous menstruating-age | Strict Herzog             | 3 months            | Windowed Herzog thresholds  | 45,198               | 22,901                 | 50.7% (50.2, 51.1)           | 9.6%                  |
| heterogeneous menstruating-age | Luteal-anchored ovulatory | 36-month full diary | Windowed Herzog C1/C2 union | 49,587               | 6,077                  | 12.3% (12.0, 12.5)           | 0.8%                  |
| heterogeneous menstruating-age | Strict Herzog             | 36-month full diary | Windowed Herzog C1/C2 union | 49,587               | 5,787                  | 11.7% (11.4, 12.0)           | 0.8%                  |
| heterogeneous menstruating-age | Luteal-anchored ovulatory | 36-month full diary | Windowed Herzog C3 only     | 39,331               | 10,534                 | 26.8% (26.3, 27.2)           | 21.3%                 |
| heterogeneous menstruating-age | Strict Herzog             | 36-month full diary | Windowed Herzog C3 only     | 39,310               | 14,433                 | 36.7% (36.2, 37.2)           | 21.4%                 |
| heterogeneous menstruating-age | Luteal-anchored ovulatory | 36-month full diary | Windowed Herzog thresholds  | 49,587               | 14,757                 | 29.8% (29.4, 30.2)           | 0.8%                  |
| heterogeneous menstruating-age | Strict Herzog             | 36-month full diary | Windowed Herzog thresholds  | 49,587               | 17,929                 | 36.2% (35.7, 36.6)           | 0.8%                  |

**Table 4 caption.** Full-diary and 3-month windowed Herzog results under strict Herzog and luteal-anchored ovulatory phase labeling.

## Table 5. Null study-level prevalence benchmarks

**Why this table is included.** This table maps person-level false positives into apparent prevalence in illustrative studies of 30, 50, and 100 participants. It reports prevalence among all participants, prevalence among classifiable participants only, and the probability of exceeding the 39.1% and 44.2% benchmark values.

**Code to call.**

In [7]:
study_benchmarks = summary_tables[
    (summary_tables.table_type == "study_level_3month")
    & (summary_tables.phase_mode == "strict_herzog")
    & (summary_tables.definition.isin([
        "A_windowed_any", "B_minimum_data_C1_or_C2",
        "C_reproducibility_C1_or_C2", "D_nb_regression_C1_or_C2"
    ]))
].copy()
study_benchmarks


,table_type,subset,cohort,window_type,window_value,definition,phase_mode,assumption_based_historical,n_windows,n_classifiable,...,indeterminate_rate,positive_rate_all_attempted,unstable_denominator,interpretation_note,n_participants,pattern_category,indeterminate_reason,n_indeterminate,p_prevalence_ge_39_1,p_prevalence_ge_44_2
19455,study_level_3month,apparent_prevalence_all,healthy_ovulatory,study_mc_calendar,3,A_windowed_any,strict_herzog,False,10000,10000.0,...,NaN,0.377010,False,NaN,30.0,NaN,NaN,NaN,0.4661,0.2058
19456,study_level_3month,apparent_prevalence_classifiable,healthy_ovulatory,study_mc_calendar,3,A_windowed_any,strict_herzog,False,10000,10000.0,...,NaN,0.416186,False,NaN,30.0,NaN,NaN,NaN,0.6120,0.3985
19457,study_level_3month,apparent_prevalence_all,healthy_ovulatory,study_mc_calendar,3,A_windowed_any,strict_herzog,False,10000,10000.0,...,NaN,0.377448,False,NaN,50.0,NaN,NaN,NaN,0.4225,0.1452
19458,study_level_3month,apparent_prevalence_classifiable,healthy_ovulatory,study_mc_calendar,3,A_windowed_any,strict_herzog,False,10000,10000.0,...,NaN,0.417180,False,NaN,50.0,NaN,NaN,NaN,0.6423,0.3640
19459,study_level_3month,apparent_prevalence_all,healthy_ovulatory,study_mc_calendar,3,A_windowed_any,strict_herzog,False,10000,10000.0,...,NaN,0.376375,False,NaN,100.0,NaN,NaN,NaN,0.3466,0.0797
19460,study_level_3month,apparent_prevalence_classifiable,healthy_ovulatory,study_mc_calendar,3,A_windowed_any,strict_herzog,False,10000,10000.0,...,NaN,0.415897,False,NaN,100.0,NaN,NaN,NaN,0.6836,0.3052
19467,study_level_3month,apparent_prevalence_all,healthy_ovulatory,study_mc_calendar,3,B_minimum_data_C1_or_C2,strict_herzog,False,10000,10000.0,...,NaN,0.000000,False,NaN,30.0,NaN,NaN,NaN,0.0000,0.0000
19468,study_level_3month,apparent_prevalence_classifiable,healthy_ovulatory,study_mc_calendar,3,B_minimum_data_C1_or_C2,strict_herzog,False,0,0.0,...,NaN,NaN,False,NaN,30.0,NaN,NaN,NaN,NaN,NaN
19469,study_level_3month,apparent_prevalence_all,healthy_ovulatory,study_mc_calendar,3,B_minimum_data_C1_or_C2,strict_herzog,False,10000,10000.0,...,NaN,0.000000,False,NaN,50.0,NaN,NaN,NaN,0.0000,0.0000
19470,study_level_3month,apparent_prevalence_classifiable,healthy_ovulatory,study_mc_calendar,3,B_minimum_data_C1_or_C2,strict_herzog,False,0,0.0,...,NaN,NaN,False,NaN,50.0,NaN,NaN,NaN,NaN,NaN


| Cohort                         | CE definition                             | Participants per study | Analysis denominator           | Monte Carlo studies | Mean apparent CE prevalence | 2.5th percentile | 97.5th percentile | Probability prevalence at least 39.1% | Probability prevalence at least 44.2% |
| ------------------------------ | ----------------------------------------- | ---------------------- | ------------------------------ | ------------------- | --------------------------- | ---------------- | ----------------- | ------------------------------------- | ------------------------------------- |
| healthy ovulatory              | Windowed Herzog thresholds                | 30.0                   | All participants               | 10,000              | 37.7%                       | 20.0%            | 56.7%             | 46.6%                                 | 20.6%                                 |
| healthy ovulatory              | Windowed Herzog thresholds                | 30.0                   | Classifiable participants only | 10,000              | 41.6%                       | 23.1%            | 60.0%             | 61.2%                                 | 39.9%                                 |
| healthy ovulatory              | Windowed Herzog thresholds                | 50.0                   | All participants               | 10,000              | 37.7%                       | 24.0%            | 52.0%             | 42.2%                                 | 14.5%                                 |
| healthy ovulatory              | Windowed Herzog thresholds                | 50.0                   | Classifiable participants only | 10,000              | 41.7%                       | 27.7%            | 56.1%             | 64.2%                                 | 36.4%                                 |
| healthy ovulatory              | Windowed Herzog thresholds                | 100.0                  | All participants               | 10,000              | 37.6%                       | 28.0%            | 47.0%             | 34.7%                                 | 8.0%                                  |
| healthy ovulatory              | Windowed Herzog thresholds                | 100.0                  | Classifiable participants only | 10,000              | 41.6%                       | 31.8%            | 51.7%             | 68.4%                                 | 30.5%                                 |
| healthy ovulatory              | Windowed Herzog C1/C2 with minimum data   | 30.0                   | All participants               | 10,000              | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| healthy ovulatory              | Windowed Herzog C1/C2 with minimum data   | 30.0                   | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| healthy ovulatory              | Windowed Herzog C1/C2 with minimum data   | 50.0                   | All participants               | 10,000              | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| healthy ovulatory              | Windowed Herzog C1/C2 with minimum data   | 50.0                   | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| healthy ovulatory              | Windowed Herzog C1/C2 with minimum data   | 100.0                  | All participants               | 10,000              | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| healthy ovulatory              | Windowed Herzog C1/C2 with minimum data   | 100.0                  | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| healthy ovulatory              | Cycle reproducibility C1/C2, 6-cycle rule | 30.0                   | All participants               | 10,000              | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| healthy ovulatory              | Cycle reproducibility C1/C2, 6-cycle rule | 30.0                   | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| healthy ovulatory              | Cycle reproducibility C1/C2, 6-cycle rule | 50.0                   | All participants               | 10,000              | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| healthy ovulatory              | Cycle reproducibility C1/C2, 6-cycle rule | 50.0                   | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| healthy ovulatory              | Cycle reproducibility C1/C2, 6-cycle rule | 100.0                  | All participants               | 10,000              | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| healthy ovulatory              | Cycle reproducibility C1/C2, 6-cycle rule | 100.0                  | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| healthy ovulatory              | Negative-binomial regression C1/C2        | 30.0                   | All participants               | 10,000              | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| healthy ovulatory              | Negative-binomial regression C1/C2        | 30.0                   | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| healthy ovulatory              | Negative-binomial regression C1/C2        | 50.0                   | All participants               | 10,000              | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| healthy ovulatory              | Negative-binomial regression C1/C2        | 50.0                   | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| healthy ovulatory              | Negative-binomial regression C1/C2        | 100.0                  | All participants               | 10,000              | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| healthy ovulatory              | Negative-binomial regression C1/C2        | 100.0                  | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| heterogeneous menstruating-age | Windowed Herzog thresholds                | 30.0                   | All participants               | 10,000              | 46.1%                       | 30.0%            | 63.3%             | 80.2%                                 | 54.6%                                 |
| heterogeneous menstruating-age | Windowed Herzog thresholds                | 30.0                   | Classifiable participants only | 10,000              | 50.8%                       | 32.1%            | 69.2%             | 89.2%                                 | 76.2%                                 |
| heterogeneous menstruating-age | Windowed Herzog thresholds                | 50.0                   | All participants               | 10,000              | 46.1%                       | 32.0%            | 60.0%             | 84.6%                                 | 55.9%                                 |
| heterogeneous menstruating-age | Windowed Herzog thresholds                | 50.0                   | Classifiable participants only | 10,000              | 50.8%                       | 36.2%            | 65.2%             | 94.5%                                 | 81.7%                                 |
| heterogeneous menstruating-age | Windowed Herzog thresholds                | 100.0                  | All participants               | 10,000              | 46.0%                       | 37.0%            | 56.0%             | 90.4%                                 | 61.8%                                 |
| heterogeneous menstruating-age | Windowed Herzog thresholds                | 100.0                  | Classifiable participants only | 10,000              | 50.8%                       | 40.7%            | 61.1%             | 98.7%                                 | 89.3%                                 |
| heterogeneous menstruating-age | Windowed Herzog C1/C2 with minimum data   | 30.0                   | All participants               | 10,000              | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| heterogeneous menstruating-age | Windowed Herzog C1/C2 with minimum data   | 30.0                   | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| heterogeneous menstruating-age | Windowed Herzog C1/C2 with minimum data   | 50.0                   | All participants               | 10,000              | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| heterogeneous menstruating-age | Windowed Herzog C1/C2 with minimum data   | 50.0                   | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| heterogeneous menstruating-age | Windowed Herzog C1/C2 with minimum data   | 100.0                  | All participants               | 10,000              | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| heterogeneous menstruating-age | Windowed Herzog C1/C2 with minimum data   | 100.0                  | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| heterogeneous menstruating-age | Cycle reproducibility C1/C2, 6-cycle rule | 30.0                   | All participants               | 10,000              | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| heterogeneous menstruating-age | Cycle reproducibility C1/C2, 6-cycle rule | 30.0                   | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| heterogeneous menstruating-age | Cycle reproducibility C1/C2, 6-cycle rule | 50.0                   | All participants               | 10,000              | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| heterogeneous menstruating-age | Cycle reproducibility C1/C2, 6-cycle rule | 50.0                   | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| heterogeneous menstruating-age | Cycle reproducibility C1/C2, 6-cycle rule | 100.0                  | All participants               | 10,000              | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| heterogeneous menstruating-age | Cycle reproducibility C1/C2, 6-cycle rule | 100.0                  | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| heterogeneous menstruating-age | Negative-binomial regression C1/C2        | 30.0                   | All participants               | 10,000              | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| heterogeneous menstruating-age | Negative-binomial regression C1/C2        | 30.0                   | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| heterogeneous menstruating-age | Negative-binomial regression C1/C2        | 50.0                   | All participants               | 10,000              | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| heterogeneous menstruating-age | Negative-binomial regression C1/C2        | 50.0                   | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| heterogeneous menstruating-age | Negative-binomial regression C1/C2        | 100.0                  | All participants               | 10,000              | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| heterogeneous menstruating-age | Negative-binomial regression C1/C2        | 100.0                  | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |

**Table 5 caption.** Study-level Monte Carlo summary from null studies using 3-month windows. The interval columns are the 2.5th and 97.5th percentiles of study-level apparent prevalence.

## Table 6. Trial-like conditioned subsets

**Why this table is included.** These subsets answer whether common enrollment restrictions reduce false positives or mainly change the classifiable denominator. The common-classifiable subset supports head-to-head comparisons because every listed core definition is defined on the same windows.

**Code to call.**

In [8]:
trial_like_subsets = summary_tables[
    (summary_tables.table_type == "window_false_positive")
    & (summary_tables.phase_mode == "strict_herzog")
    & (summary_tables.window_type == "full")
    & (summary_tables.subset.isin([
        "ge_1_seizure_day_per_month",
        "ge_2_seizures_per_month",
        "strict_23_35_day_cycles_only",
        "common_classifiable_subset",
    ]))
    & (summary_tables.definition.isin([
        "A_windowed_any", "A_windowed_C1_or_C2",
        "B_minimum_data_C1_or_C2",
        "C_reproducibility_C1_or_C2", "D_nb_regression_C1_or_C2"
    ]))
].copy()
trial_like_subsets


,table_type,subset,cohort,window_type,window_value,definition,phase_mode,assumption_based_historical,n_windows,n_classifiable,...,indeterminate_rate,positive_rate_all_attempted,unstable_denominator,interpretation_note,n_participants,pattern_category,indeterminate_reason,n_indeterminate,p_prevalence_ge_39_1,p_prevalence_ge_44_2
1273,window_false_positive,ge_1_seizure_day_per_month,healthy_ovulatory,full,full_diary,A_windowed_any,strict_herzog,False,37309,37309.0,...,0.000000,0.056072,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1299,window_false_positive,ge_1_seizure_day_per_month,population,full,full_diary,A_windowed_any,strict_herzog,False,37196,37196.0,...,0.000000,0.314846,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1377,window_false_positive,ge_1_seizure_day_per_month,healthy_ovulatory,full,full_diary,A_windowed_C1_or_C2,strict_herzog,False,37309,37309.0,...,0.000000,0.056072,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1403,window_false_positive,ge_1_seizure_day_per_month,population,full,full_diary,A_windowed_C1_or_C2,strict_herzog,False,37196,37196.0,...,0.000000,0.056915,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1689,window_false_positive,ge_1_seizure_day_per_month,healthy_ovulatory,full,full_diary,B_minimum_data_C1_or_C2,strict_herzog,False,37309,37309.0,...,0.000000,0.056072,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1715,window_false_positive,ge_1_seizure_day_per_month,population,full,full_diary,B_minimum_data_C1_or_C2,strict_herzog,False,37196,37196.0,...,0.000000,0.056915,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1793,window_false_positive,ge_1_seizure_day_per_month,healthy_ovulatory,full,full_diary,C_reproducibility_C1_or_C2,strict_herzog,False,37309,6318.0,...,0.830657,0.000000,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1819,window_false_positive,ge_1_seizure_day_per_month,population,full,full_diary,C_reproducibility_C1_or_C2,strict_herzog,False,37196,9216.0,...,0.752231,0.000027,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2001,window_false_positive,ge_1_seizure_day_per_month,healthy_ovulatory,full,full_diary,D_nb_regression_C1_or_C2,strict_herzog,False,37309,37309.0,...,0.000000,0.045780,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2027,window_false_positive,ge_1_seizure_day_per_month,population,full,full_diary,D_nb_regression_C1_or_C2,strict_herzog,False,37196,37196.0,...,0.000000,0.044844,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN


| Cohort                         | Analysis denominator             | CE definition                             | Classifiable windows | False-positive windows | False-positive rate (95% CI) | Indeterminate windows |
| ------------------------------ | -------------------------------- | ----------------------------------------- | -------------------- | ---------------------- | ---------------------------- | --------------------- |
| healthy ovulatory              | Common classifiable subset       | Windowed Herzog C1/C2 union               | 6,318                | 153                    | 2.4% (2.1, 2.8)              | 0.0%                  |
| healthy ovulatory              | Common classifiable subset       | Windowed Herzog thresholds                | 6,318                | 153                    | 2.4% (2.1, 2.8)              | 0.0%                  |
| healthy ovulatory              | Common classifiable subset       | Windowed Herzog C1/C2 with minimum data   | 6,318                | 153                    | 2.4% (2.1, 2.8)              | 0.0%                  |
| healthy ovulatory              | Common classifiable subset       | Cycle reproducibility C1/C2, 6-cycle rule | 6,318                | 0                      | 0.0% (0.0, 0.1)              | 0.0%                  |
| healthy ovulatory              | Common classifiable subset       | Negative-binomial regression C1/C2        | 6,318                | 155                    | 2.5% (2.1, 2.9)              | 0.0%                  |
| healthy ovulatory              | At least 1 seizure day per month | Windowed Herzog C1/C2 union               | 37,309               | 2,092                  | 5.6% (5.4, 5.8)              | 0.0%                  |
| healthy ovulatory              | At least 1 seizure day per month | Windowed Herzog thresholds                | 37,309               | 2,092                  | 5.6% (5.4, 5.8)              | 0.0%                  |
| healthy ovulatory              | At least 1 seizure day per month | Windowed Herzog C1/C2 with minimum data   | 37,309               | 2,092                  | 5.6% (5.4, 5.8)              | 0.0%                  |
| healthy ovulatory              | At least 1 seizure day per month | Cycle reproducibility C1/C2, 6-cycle rule | 6,318                | 0                      | 0.0% (0.0, 0.1)              | 83.1%                 |
| healthy ovulatory              | At least 1 seizure day per month | Negative-binomial regression C1/C2        | 37,309               | 1,708                  | 4.6% (4.4, 4.8)              | 0.0%                  |
| healthy ovulatory              | At least 2 seizures per month    | Windowed Herzog C1/C2 union               | 30,106               | 1,303                  | 4.3% (4.1, 4.6)              | 0.0%                  |
| healthy ovulatory              | At least 2 seizures per month    | Windowed Herzog thresholds                | 30,106               | 1,303                  | 4.3% (4.1, 4.6)              | 0.0%                  |
| healthy ovulatory              | At least 2 seizures per month    | Windowed Herzog C1/C2 with minimum data   | 30,106               | 1,303                  | 4.3% (4.1, 4.6)              | 0.0%                  |
| healthy ovulatory              | At least 2 seizures per month    | Cycle reproducibility C1/C2, 6-cycle rule | 6,318                | 0                      | 0.0% (0.0, 0.1)              | 79.0%                 |
| healthy ovulatory              | At least 2 seizures per month    | Negative-binomial regression C1/C2        | 30,106               | 1,353                  | 4.5% (4.3, 4.7)              | 0.0%                  |
| healthy ovulatory              | Strict 23-35 day cycles only     | Windowed Herzog C1/C2 union               | 5,301                | 575                    | 10.8% (10.0, 11.7)           | 0.9%                  |
| healthy ovulatory              | Strict 23-35 day cycles only     | Windowed Herzog thresholds                | 5,301                | 575                    | 10.8% (10.0, 11.7)           | 0.9%                  |
| healthy ovulatory              | Strict 23-35 day cycles only     | Windowed Herzog C1/C2 with minimum data   | 5,172                | 506                    | 9.8% (9.0, 10.6)             | 3.3%                  |
| healthy ovulatory              | Strict 23-35 day cycles only     | Cycle reproducibility C1/C2, 6-cycle rule | 767                  | 0                      | 0.0% (0.0, 0.5)              | 85.7%                 |
| healthy ovulatory              | Strict 23-35 day cycles only     | Negative-binomial regression C1/C2        | 5,172                | 208                    | 4.0% (3.5, 4.6)              | 3.3%                  |
| heterogeneous menstruating-age | Common classifiable subset       | Windowed Herzog C1/C2 union               | 13,341               | 611                    | 4.6% (4.2, 4.9)              | 0.0%                  |
| heterogeneous menstruating-age | Common classifiable subset       | Windowed Herzog thresholds                | 13,341               | 3,499                  | 26.2% (25.5, 27.0)           | 0.0%                  |
| heterogeneous menstruating-age | Common classifiable subset       | Windowed Herzog C1/C2 with minimum data   | 13,341               | 611                    | 4.6% (4.2, 4.9)              | 0.0%                  |
| heterogeneous menstruating-age | Common classifiable subset       | Cycle reproducibility C1/C2, 6-cycle rule | 9,227                | 1                      | 0.0% (0.0, 0.1)              | 30.8%                 |
| heterogeneous menstruating-age | Common classifiable subset       | Negative-binomial regression C1/C2        | 13,341               | 558                    | 4.2% (3.9, 4.5)              | 0.0%                  |
| heterogeneous menstruating-age | At least 1 seizure day per month | Windowed Herzog C1/C2 union               | 37,196               | 2,117                  | 5.7% (5.5, 5.9)              | 0.0%                  |
| heterogeneous menstruating-age | At least 1 seizure day per month | Windowed Herzog thresholds                | 37,196               | 11,711                 | 31.5% (31.0, 32.0)           | 0.0%                  |
| heterogeneous menstruating-age | At least 1 seizure day per month | Windowed Herzog C1/C2 with minimum data   | 37,196               | 2,117                  | 5.7% (5.5, 5.9)              | 0.0%                  |
| heterogeneous menstruating-age | At least 1 seizure day per month | Cycle reproducibility C1/C2, 6-cycle rule | 9,216                | 1                      | 0.0% (0.0, 0.1)              | 75.2%                 |
| heterogeneous menstruating-age | At least 1 seizure day per month | Negative-binomial regression C1/C2        | 37,196               | 1,668                  | 4.5% (4.3, 4.7)              | 0.0%                  |
| heterogeneous menstruating-age | At least 2 seizures per month    | Windowed Herzog C1/C2 union               | 30,016               | 1,321                  | 4.4% (4.2, 4.6)              | 0.0%                  |
| heterogeneous menstruating-age | At least 2 seizures per month    | Windowed Herzog thresholds                | 30,016               | 8,980                  | 29.9% (29.4, 30.4)           | 0.0%                  |
| heterogeneous menstruating-age | At least 2 seizures per month    | Windowed Herzog C1/C2 with minimum data   | 30,016               | 1,321                  | 4.4% (4.2, 4.6)              | 0.0%                  |
| heterogeneous menstruating-age | At least 2 seizures per month    | Cycle reproducibility C1/C2, 6-cycle rule | 9,149                | 1                      | 0.0% (0.0, 0.1)              | 69.5%                 |
| heterogeneous menstruating-age | At least 2 seizures per month    | Negative-binomial regression C1/C2        | 30,016               | 1,350                  | 4.5% (4.3, 4.7)              | 0.0%                  |
| heterogeneous menstruating-age | Strict 23-35 day cycles only     | Windowed Herzog C1/C2 union               | 2,896                | 318                    | 11.0% (9.9, 12.2)            | 1.2%                  |
| heterogeneous menstruating-age | Strict 23-35 day cycles only     | Windowed Herzog thresholds                | 2,896                | 1,113                  | 38.4% (36.7, 40.2)           | 1.2%                  |
| heterogeneous menstruating-age | Strict 23-35 day cycles only     | Windowed Herzog C1/C2 with minimum data   | 2,825                | 283                    | 10.0% (9.0, 11.2)            | 3.6%                  |
| heterogeneous menstruating-age | Strict 23-35 day cycles only     | Cycle reproducibility C1/C2, 6-cycle rule | 427                  | 0                      | 0.0% (0.0, 0.9)              | 85.4%                 |
| heterogeneous menstruating-age | Strict 23-35 day cycles only     | Negative-binomial regression C1/C2        | 2,825                | 111                    | 3.9% (3.3, 4.7)              | 3.6%                  |

**Table 6 caption.** Full-diary false-positive rates after applying trial-like eligibility restrictions or a common classifiable denominator. This separates changes in apparent risk from changes in analyzability.

## Table 7. C1/C2/C3 decomposition and C3 exclusion

**Why this table is included.** This table directly addresses whether the heterogeneous-cohort signal is driven by C3 logic. It reports mutually exclusive pattern categories and C1/C2 union comparisons for full-diary windows.

**Code to call.**

In [9]:
pattern_decomposition = summary_tables[
    (summary_tables.table_type == "pattern_decomposition")
    & (summary_tables.phase_mode == "strict_herzog")
    & (summary_tables.window_type == "full")
    & (summary_tables.definition.isin(["A_windowed", "B_minimum_data", "D_nb_regression"]))
].copy()
pattern_decomposition


,table_type,subset,cohort,window_type,window_value,definition,phase_mode,assumption_based_historical,n_windows,n_classifiable,...,indeterminate_rate,positive_rate_all_attempted,unstable_denominator,interpretation_note,n_participants,pattern_category,indeterminate_reason,n_indeterminate,p_prevalence_ge_39_1,p_prevalence_ge_44_2
18044,pattern_decomposition,mutually_exclusive_patterns,healthy_ovulatory,full,full_diary,A_windowed,strict_herzog,NaN,50000,49605.0,...,0.00790,0.05928,False,NaN,NaN,C1 only,NaN,NaN,NaN,NaN
18045,pattern_decomposition,mutually_exclusive_patterns,healthy_ovulatory,full,full_diary,A_windowed,strict_herzog,NaN,50000,49605.0,...,0.00790,0.03632,False,NaN,NaN,C2 only,NaN,NaN,NaN,NaN
18046,pattern_decomposition,mutually_exclusive_patterns,healthy_ovulatory,full,full_diary,A_windowed,strict_herzog,NaN,50000,49605.0,...,0.00790,0.02032,False,NaN,NaN,C1+C2,NaN,NaN,NaN,NaN
18047,pattern_decomposition,mutually_exclusive_patterns,healthy_ovulatory,full,full_diary,A_windowed,strict_herzog,NaN,50000,49605.0,...,0.00790,0.00000,False,NaN,NaN,C3 only,NaN,NaN,NaN,NaN
18048,pattern_decomposition,mutually_exclusive_patterns,healthy_ovulatory,full,full_diary,A_windowed,strict_herzog,NaN,50000,49605.0,...,0.00790,0.00000,False,NaN,NaN,C3 plus C1/C2,NaN,NaN,NaN,NaN
18049,pattern_decomposition,mutually_exclusive_patterns,healthy_ovulatory,full,full_diary,A_windowed,strict_herzog,NaN,50000,49605.0,...,0.00790,0.87618,False,NaN,NaN,none,NaN,NaN,NaN,NaN
18200,pattern_decomposition,mutually_exclusive_patterns,population,full,full_diary,A_windowed,strict_herzog,NaN,50000,49587.0,...,0.00826,0.03854,False,NaN,NaN,C1 only,NaN,NaN,NaN,NaN
18201,pattern_decomposition,mutually_exclusive_patterns,population,full,full_diary,A_windowed,strict_herzog,NaN,50000,49587.0,...,0.00826,0.02010,False,NaN,NaN,C2 only,NaN,NaN,NaN,NaN
18202,pattern_decomposition,mutually_exclusive_patterns,population,full,full_diary,A_windowed,strict_herzog,NaN,50000,49587.0,...,0.00826,0.01128,False,NaN,NaN,C1+C2,NaN,NaN,NaN,NaN
18203,pattern_decomposition,mutually_exclusive_patterns,population,full,full_diary,A_windowed,strict_herzog,NaN,50000,49587.0,...,0.00826,0.24284,False,NaN,NaN,C3 only,NaN,NaN,NaN,NaN


| Cohort                         | CE definition   | Pattern category | Classifiable windows | False-positive windows | False-positive rate | Rate among all attempted | Indeterminate windows |
| ------------------------------ | --------------- | ---------------- | -------------------- | ---------------------- | ------------------- | ------------------------ | --------------------- |
| healthy ovulatory              | A_windowed      | C1 only          | 49,605               | 2,964                  | 6.0%                | 5.9%                     | 0.8%                  |
| healthy ovulatory              | A_windowed      | C1+C2            | 49,605               | 1,016                  | 2.0%                | 2.0%                     | 0.8%                  |
| healthy ovulatory              | A_windowed      | C2 only          | 49,605               | 1,816                  | 3.7%                | 3.6%                     | 0.8%                  |
| healthy ovulatory              | A_windowed      | C3 only          | 49,605               | 0                      | 0.0%                | 0.0%                     | 0.8%                  |
| healthy ovulatory              | A_windowed      | C3 plus C1/C2    | 49,605               | 0                      | 0.0%                | 0.0%                     | 0.8%                  |
| healthy ovulatory              | A_windowed      | none             | 49,605               | 43,809                 | 88.3%               | 87.6%                    | 0.8%                  |
| healthy ovulatory              | B_minimum_data  | C1 only          | 48,359               | 2,725                  | 5.6%                | 5.5%                     | 3.3%                  |
| healthy ovulatory              | B_minimum_data  | C1+C2            | 48,359               | 894                    | 1.8%                | 1.8%                     | 3.3%                  |
| healthy ovulatory              | B_minimum_data  | C2 only          | 48,359               | 1,535                  | 3.2%                | 3.1%                     | 3.3%                  |
| healthy ovulatory              | B_minimum_data  | C3 only          | 48,359               | 0                      | 0.0%                | 0.0%                     | 3.3%                  |
| healthy ovulatory              | B_minimum_data  | C3 plus C1/C2    | 48,359               | 0                      | 0.0%                | 0.0%                     | 3.3%                  |
| healthy ovulatory              | B_minimum_data  | none             | 48,359               | 43,205                 | 89.3%               | 86.4%                    | 3.3%                  |
| healthy ovulatory              | D_nb_regression | C1 only          | 48,359               | 1,020                  | 2.1%                | 2.0%                     | 3.3%                  |
| healthy ovulatory              | D_nb_regression | C1+C2            | 48,359               | 238                    | 0.5%                | 0.5%                     | 3.3%                  |
| healthy ovulatory              | D_nb_regression | C2 only          | 48,359               | 867                    | 1.8%                | 1.7%                     | 3.3%                  |
| healthy ovulatory              | D_nb_regression | C3 only          | 48,359               | 0                      | 0.0%                | 0.0%                     | 3.3%                  |
| healthy ovulatory              | D_nb_regression | C3 plus C1/C2    | 48,359               | 0                      | 0.0%                | 0.0%                     | 3.3%                  |
| healthy ovulatory              | D_nb_regression | none             | 48,359               | 46,234                 | 95.6%               | 92.5%                    | 3.3%                  |
| heterogeneous menstruating-age | A_windowed      | C1 only          | 49,587               | 1,927                  | 3.9%                | 3.9%                     | 0.8%                  |
| heterogeneous menstruating-age | A_windowed      | C1+C2            | 49,587               | 564                    | 1.1%                | 1.1%                     | 0.8%                  |
| heterogeneous menstruating-age | A_windowed      | C2 only          | 49,587               | 1,005                  | 2.0%                | 2.0%                     | 0.8%                  |
| heterogeneous menstruating-age | A_windowed      | C3 only          | 49,587               | 12,142                 | 24.5%               | 24.3%                    | 0.8%                  |
| heterogeneous menstruating-age | A_windowed      | C3 plus C1/C2    | 49,587               | 2,291                  | 4.6%                | 4.6%                     | 0.8%                  |
| heterogeneous menstruating-age | A_windowed      | none             | 49,587               | 31,658                 | 63.8%               | 63.3%                    | 0.8%                  |
| heterogeneous menstruating-age | B_minimum_data  | C1 only          | 48,333               | 1,727                  | 3.6%                | 3.5%                     | 3.3%                  |
| heterogeneous menstruating-age | B_minimum_data  | C1+C2            | 48,333               | 491                    | 1.0%                | 1.0%                     | 3.3%                  |
| heterogeneous menstruating-age | B_minimum_data  | C2 only          | 48,333               | 814                    | 1.7%                | 1.6%                     | 3.3%                  |
| heterogeneous menstruating-age | B_minimum_data  | C3 only          | 48,333               | 12,034                 | 24.9%               | 24.1%                    | 3.3%                  |
| heterogeneous menstruating-age | B_minimum_data  | C3 plus C1/C2    | 48,333               | 2,115                  | 4.4%                | 4.2%                     | 3.3%                  |
| heterogeneous menstruating-age | B_minimum_data  | none             | 48,333               | 31,152                 | 64.5%               | 62.3%                    | 3.3%                  |
| heterogeneous menstruating-age | D_nb_regression | C1 only          | 48,333               | 1,020                  | 2.1%                | 2.0%                     | 3.3%                  |
| heterogeneous menstruating-age | D_nb_regression | C1+C2            | 48,333               | 270                    | 0.6%                | 0.5%                     | 3.3%                  |
| heterogeneous menstruating-age | D_nb_regression | C2 only          | 48,333               | 745                    | 1.5%                | 1.5%                     | 3.3%                  |
| heterogeneous menstruating-age | D_nb_regression | C3 only          | 48,333               | 0                      | 0.0%                | 0.0%                     | 3.3%                  |
| heterogeneous menstruating-age | D_nb_regression | C3 plus C1/C2    | 48,333               | 0                      | 0.0%                | 0.0%                     | 3.3%                  |
| heterogeneous menstruating-age | D_nb_regression | none             | 48,333               | 46,298                 | 95.8%               | 92.6%                    | 3.3%                  |

**Table 7 caption.** Full-diary pattern decomposition and C3-exclusion sensitivity. C3 is evaluated only when ILP logic is applicable.

## Table 8. Negative-binomial dispersion sensitivity

**Why this table is included.** This table separates the full-diary stabilized-dispersion regression comparator from a full-window, window-only dispersion sensitivity.

**Code to call.**

In [10]:
nb_dispersion_sensitivity = summary_tables[
    (summary_tables.table_type == "window_false_positive")
    & (summary_tables.phase_mode == "strict_herzog")
    & (summary_tables.subset == "all")
    & (summary_tables.window_type == "full")
    & (summary_tables.definition.isin([
        "D_nb_regression_C1_or_C2", "D_nb_regression_window_alpha_C1_or_C2"
    ]))
].copy()
nb_dispersion_sensitivity


,table_type,subset,cohort,window_type,window_value,definition,phase_mode,assumption_based_historical,n_windows,n_classifiable,...,indeterminate_rate,positive_rate_all_attempted,unstable_denominator,interpretation_note,n_participants,pattern_category,indeterminate_reason,n_indeterminate,p_prevalence_ge_39_1,p_prevalence_ge_44_2
805,window_false_positive,all,healthy_ovulatory,full,full_diary,D_nb_regression_C1_or_C2,strict_herzog,False,50000,48359.0,...,0.03282,0.0425,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
831,window_false_positive,all,population,full,full_diary,D_nb_regression_C1_or_C2,strict_herzog,False,50000,48333.0,...,0.03334,0.0407,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
909,window_false_positive,all,healthy_ovulatory,full,full_diary,D_nb_regression_window_alpha_C1_or_C2,strict_herzog,False,50000,48359.0,...,0.03282,0.0425,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
935,window_false_positive,all,population,full,full_diary,D_nb_regression_window_alpha_C1_or_C2,strict_herzog,False,50000,48333.0,...,0.03334,0.0407,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN


| Cohort                         | Observation window  | CE definition                                              | Classifiable windows | False-positive windows | False-positive rate (95% CI) | Indeterminate windows |
| ------------------------------ | ------------------- | ---------------------------------------------------------- | -------------------- | ---------------------- | ---------------------------- | --------------------- |
| healthy ovulatory              | 36-month full diary | Negative-binomial regression C1/C2                         | 48,359               | 2,125                  | 4.4% (4.2, 4.6)              | 3.3%                  |
| healthy ovulatory              | 36-month full diary | Negative-binomial regression C1/C2, window-only dispersion | 48,359               | 2,125                  | 4.4% (4.2, 4.6)              | 3.3%                  |
| heterogeneous menstruating-age | 36-month full diary | Negative-binomial regression C1/C2                         | 48,333               | 2,035                  | 4.2% (4.0, 4.4)              | 3.3%                  |
| heterogeneous menstruating-age | 36-month full diary | Negative-binomial regression C1/C2, window-only dispersion | 48,333               | 2,035                  | 4.2% (4.0, 4.4)              | 3.3%                  |

**Table 8 caption.** Negative-binomial apparent classification rates in full-diary windows using full-diary stabilized alpha and window-only alpha. Both use the same M/O model and Holm family.

## Table 9. Seizure-burden and cycle-regularity strata

**Why this table is included.** The requested strata diagnose where false positives concentrate. Seizure-frequency strata use observed full-diary seizure-days per month, and window-seizure-day strata use total seizure days within each analyzed window.

**Code to call.**

In [11]:
strata_rows = summary_tables[
    (summary_tables.table_type == "window_false_positive")
    & (summary_tables.phase_mode == "strict_herzog")
    & (summary_tables.definition.isin(["A_windowed_any", "A_windowed_C1_or_C2", "B_minimum_data_C1_or_C2", "D_nb_regression_C1_or_C2"]))
    & (
        summary_tables.subset.astype(str).str.startswith("seizure_frequency:")
        | summary_tables.subset.astype(str).str.startswith("cycle_regularity:")
        | summary_tables.subset.astype(str).str.startswith("window_seizure_days_")
    )
].copy()
strata_rows


,table_type,subset,cohort,window_type,window_value,definition,phase_mode,assumption_based_historical,n_windows,n_classifiable,...,indeterminate_rate,positive_rate_all_attempted,unstable_denominator,interpretation_note,n_participants,pattern_category,indeterminate_reason,n_indeterminate,p_prevalence_ge_39_1,p_prevalence_ge_44_2
4895,window_false_positive,window_seizure_days_0_to_3,healthy_ovulatory,calendar,1,A_windowed_any,strict_herzog,False,35302,23516.0,...,0.333862,0.329868,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4896,window_false_positive,window_seizure_days_0_to_3,healthy_ovulatory,calendar,3,A_windowed_any,strict_herzog,False,15576,10897.0,...,0.300398,0.352979,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4897,window_false_positive,window_seizure_days_0_to_3,healthy_ovulatory,calendar,4,A_windowed_any,strict_herzog,False,12373,8675.0,...,0.298877,0.360462,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4898,window_false_positive,window_seizure_days_0_to_3,healthy_ovulatory,calendar,6,A_windowed_any,strict_herzog,False,8860,6393.0,...,0.278442,0.369413,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4899,window_false_positive,window_seizure_days_0_to_3,healthy_ovulatory,calendar,9,A_windowed_any,strict_herzog,False,6195,4526.0,...,0.269411,0.371106,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16329,window_false_positive,cycle_regularity:SD cycle length >=4 days,population,calendar,36,D_nb_regression_C1_or_C2,strict_herzog,False,24812,0.0,...,1.000000,0.000000,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16330,window_false_positive,cycle_regularity:SD cycle length >=4 days,population,cycle,3,D_nb_regression_C1_or_C2,strict_herzog,False,24812,0.0,...,1.000000,0.000000,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16331,window_false_positive,cycle_regularity:SD cycle length >=4 days,population,cycle,6,D_nb_regression_C1_or_C2,strict_herzog,False,24812,0.0,...,1.000000,0.000000,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16332,window_false_positive,cycle_regularity:SD cycle length >=4 days,population,cycle,12,D_nb_regression_C1_or_C2,strict_herzog,False,24812,0.0,...,1.000000,0.000000,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN


| Cohort                         | Stratum type               | Stratum                           | CE definition                           | Classifiable windows | False-positive windows | False-positive rate | Indeterminate windows |
| ------------------------------ | -------------------------- | --------------------------------- | --------------------------------------- | -------------------- | ---------------------- | ------------------- | --------------------- |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 30,902               | 14,718                 | 47.6%               | 23.6%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 36,676               | 15,167                 | 41.4%               | 9.3%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 37,471               | 14,750                 | 39.4%               | 7.3%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 38,451               | 13,103                 | 34.1%               | 4.9%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 39,101               | 11,535                 | 29.5%               | 3.3%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 39,436               | 10,082                 | 25.6%               | 2.5%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 39,769               | 7,903                  | 19.9%               | 1.7%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 39,956               | 6,352                  | 15.9%               | 1.2%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 40,135               | 4,632                  | 11.5%               | 0.8%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 36,525               | 15,054                 | 41.2%               | 9.7%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 38,353               | 13,198                 | 34.4%               | 5.2%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 39,388               | 9,959                  | 25.3%               | 2.6%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 40,135               | 4,632                  | 11.5%               | 0.8%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 30,902               | 14,718                 | 47.6%               | 23.6%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 36,676               | 15,167                 | 41.4%               | 9.3%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 37,471               | 14,750                 | 39.4%               | 7.3%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 38,451               | 13,103                 | 34.1%               | 4.9%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 39,101               | 11,535                 | 29.5%               | 3.3%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 39,436               | 10,082                 | 25.6%               | 2.5%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 39,769               | 7,903                  | 19.9%               | 1.7%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 39,956               | 6,352                  | 15.9%               | 1.2%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 40,135               | 4,632                  | 11.5%               | 0.8%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 36,525               | 15,054                 | 41.2%               | 9.7%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 38,353               | 13,198                 | 34.4%               | 5.2%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 39,388               | 9,959                  | 25.3%               | 2.6%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 40,135               | 4,632                  | 11.5%               | 0.8%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 30,409               | 11,131                 | 36.6%               | 24.8%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 33,318               | 10,485                 | 31.5%               | 17.6%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 35,455               | 9,688                  | 27.3%               | 12.3%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 36,613               | 8,630                  | 23.6%               | 9.5%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 37,817               | 6,915                  | 18.3%               | 6.5%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 38,463               | 5,598                  | 14.6%               | 4.9%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 39,142               | 4,122                  | 10.5%               | 3.2%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 1                    | 1                      | 100.0%              | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 33,076               | 10,544                 | 31.9%               | 18.2%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 36,490               | 8,468                  | 23.2%               | 9.8%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 39,142               | 4,122                  | 10.5%               | 3.2%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 39,142               | 1,713                  | 4.4%                | 3.2%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 366                  | 177                    | 48.4%               | 23.6%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 435                  | 173                    | 39.8%               | 9.2%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 436                  | 165                    | 37.8%               | 9.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 447                  | 160                    | 35.8%               | 6.7%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 457                  | 146                    | 31.9%               | 4.6%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 463                  | 107                    | 23.1%               | 3.3%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 469                  | 115                    | 24.5%               | 2.1%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 471                  | 85                     | 18.0%               | 1.7%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 475                  | 70                     | 14.7%               | 0.8%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 428                  | 175                    | 40.9%               | 10.6%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 451                  | 166                    | 36.8%               | 5.8%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 459                  | 116                    | 25.3%               | 4.2%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 475                  | 70                     | 14.7%               | 0.8%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 366                  | 177                    | 48.4%               | 23.6%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 435                  | 173                    | 39.8%               | 9.2%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 436                  | 165                    | 37.8%               | 9.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 447                  | 160                    | 35.8%               | 6.7%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 457                  | 146                    | 31.9%               | 4.6%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 463                  | 107                    | 23.1%               | 3.3%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 469                  | 115                    | 24.5%               | 2.1%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 471                  | 85                     | 18.0%               | 1.7%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 475                  | 70                     | 14.7%               | 0.8%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 428                  | 175                    | 40.9%               | 10.6%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 451                  | 166                    | 36.8%               | 5.8%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 459                  | 116                    | 25.3%               | 4.2%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 475                  | 70                     | 14.7%               | 0.8%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 374                  | 136                    | 36.4%               | 21.9%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 391                  | 128                    | 32.7%               | 18.4%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 412                  | 118                    | 28.6%               | 14.0%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 430                  | 89                     | 20.7%               | 10.2%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 437                  | 97                     | 22.2%               | 8.8%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 446                  | 70                     | 15.7%               | 6.9%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 454                  | 59                     | 13.0%               | 5.2%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 391                  | 132                    | 33.8%               | 18.4%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 425                  | 95                     | 22.4%               | 11.3%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 454                  | 59                     | 13.0%               | 5.2%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 454                  | 24                     | 5.3%                | 5.2%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 6,946                | 3,268                  | 47.0%               | 23.5%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 8,210                | 3,418                  | 41.6%               | 9.6%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 8,395                | 3,282                  | 39.1%               | 7.5%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 8,635                | 2,960                  | 34.3%               | 4.9%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 8,773                | 2,596                  | 29.6%               | 3.4%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 8,832                | 2,253                  | 25.5%               | 2.7%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 8,921                | 1,745                  | 19.6%               | 1.7%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 8,954                | 1,432                  | 16.0%               | 1.4%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 8,995                | 1,094                  | 12.2%               | 0.9%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 8,180                | 3,309                  | 40.5%               | 9.9%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 8,621                | 2,879                  | 33.4%               | 5.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 8,829                | 2,287                  | 25.9%               | 2.7%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 8,995                | 1,094                  | 12.2%               | 0.9%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 6,946                | 3,268                  | 47.0%               | 23.5%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 8,210                | 3,418                  | 41.6%               | 9.6%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 8,395                | 3,282                  | 39.1%               | 7.5%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 8,635                | 2,960                  | 34.3%               | 4.9%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 8,773                | 2,596                  | 29.6%               | 3.4%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 8,832                | 2,253                  | 25.5%               | 2.7%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 8,921                | 1,745                  | 19.6%               | 1.7%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 8,954                | 1,432                  | 16.0%               | 1.4%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 8,995                | 1,094                  | 12.2%               | 0.9%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 8,180                | 3,309                  | 40.5%               | 9.9%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 8,621                | 2,879                  | 33.4%               | 5.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 8,829                | 2,287                  | 25.9%               | 2.7%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 8,995                | 1,094                  | 12.2%               | 0.9%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 6,844                | 2,470                  | 36.1%               | 24.6%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 7,431                | 2,337                  | 31.4%               | 18.1%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 7,938                | 2,172                  | 27.4%               | 12.6%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 8,225                | 1,935                  | 23.5%               | 9.4%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 8,469                | 1,520                  | 17.9%               | 6.7%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 8,626                | 1,281                  | 14.9%               | 5.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 8,763                | 973                    | 11.1%               | 3.5%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 15                   | 2                      | 13.3%               | 99.8%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 7,452                | 2,278                  | 30.6%               | 17.9%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 8,191                | 1,945                  | 23.7%               | 9.8%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 8,763                | 973                    | 11.1%               | 3.5%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 8,763                | 388                    | 4.4%                | 3.5%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 23,889               | 11,545                 | 48.3%               | 12.1%                 |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 27,010               | 11,192                 | 41.4%               | 0.6%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 27,124               | 10,616                 | 39.1%               | 0.2%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 27,170               | 8,957                  | 33.0%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 27,175               | 7,423                  | 27.3%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 27,175               | 6,091                  | 22.4%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 27,175               | 4,377                  | 16.1%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 27,175               | 3,156                  | 11.6%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 27,175               | 1,834                  | 6.7%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 26,948               | 11,128                 | 41.3%               | 0.8%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 27,170               | 9,053                  | 33.3%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 27,175               | 6,151                  | 22.6%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 27,175               | 1,834                  | 6.7%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 23,889               | 11,545                 | 48.3%               | 12.1%                 |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 27,010               | 11,192                 | 41.4%               | 0.6%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 27,124               | 10,616                 | 39.1%               | 0.2%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 27,170               | 8,957                  | 33.0%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 27,175               | 7,423                  | 27.3%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 27,175               | 6,091                  | 22.4%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 27,175               | 4,377                  | 16.1%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 27,175               | 3,156                  | 11.6%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 27,175               | 1,834                  | 6.7%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 26,948               | 11,128                 | 41.3%               | 0.8%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 27,170               | 9,053                  | 33.3%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 27,175               | 6,151                  | 22.6%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 27,175               | 1,834                  | 6.7%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 25,333               | 9,731                  | 38.4%               | 6.8%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 26,833               | 8,788                  | 32.8%               | 1.3%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 27,159               | 7,417                  | 27.3%               | 0.1%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 27,172               | 6,090                  | 22.4%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 27,175               | 4,377                  | 16.1%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 27,175               | 3,156                  | 11.6%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 27,175               | 1,834                  | 6.7%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 9                    | 2                      | 22.2%               | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 26,698               | 8,808                  | 33.0%               | 1.8%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 27,174               | 6,151                  | 22.6%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 27,175               | 1,834                  | 6.7%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 27,175               | 1,392                  | 5.1%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,072               | 4,489                  | 44.6%               | 0.6%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,134               | 3,485                  | 34.4%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,134               | 3,083                  | 30.4%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,134               | 2,401                  | 23.7%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,134               | 1,834                  | 18.1%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,134               | 1,364                  | 13.5%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,134               | 817                    | 8.1%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,134               | 522                    | 5.2%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,134               | 258                    | 2.5%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,134               | 3,417                  | 33.7%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,134               | 2,474                  | 24.4%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,134               | 1,381                  | 13.6%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,134               | 258                    | 2.5%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,072               | 4,489                  | 44.6%               | 0.6%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,134               | 3,485                  | 34.4%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,134               | 3,083                  | 30.4%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,134               | 2,401                  | 23.7%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,134               | 1,834                  | 18.1%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,134               | 1,364                  | 13.5%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,134               | 817                    | 8.1%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,134               | 522                    | 5.2%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,134               | 258                    | 2.5%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,134               | 3,417                  | 33.7%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,134               | 2,474                  | 24.4%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,134               | 1,381                  | 13.6%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,134               | 258                    | 2.5%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 10,132               | 3,081                  | 30.4%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 10,134               | 2,401                  | 23.7%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 10,134               | 1,834                  | 18.1%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 10,134               | 1,364                  | 13.5%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 10,134               | 817                    | 8.1%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 10,134               | 522                    | 5.2%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 10,134               | 258                    | 2.5%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 7                    | 1                      | 14.3%               | 99.9%                 |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 10,134               | 2,474                  | 24.4%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 10,134               | 1,381                  | 13.6%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 10,134               | 258                    | 2.5%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 10,134               | 316                    | 3.1%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 4,253                | 2,129                  | 50.1%               | 66.5%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 8,177                | 4,081                  | 49.9%               | 35.6%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 9,044                | 4,498                  | 49.7%               | 28.7%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 10,229               | 4,865                  | 47.6%               | 19.4%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 11,022               | 5,020                  | 45.5%               | 13.2%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 11,422               | 4,987                  | 43.7%               | 10.0%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 11,850               | 4,569                  | 38.6%               | 6.6%                  |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 12,072               | 4,191                  | 34.7%               | 4.9%                  |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 12,296               | 3,704                  | 30.1%               | 3.1%                  |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 8,051                | 3,993                  | 49.6%               | 36.6%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 10,121               | 4,716                  | 46.6%               | 20.3%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 11,367               | 4,830                  | 42.5%               | 10.4%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 12,296               | 3,704                  | 30.1%               | 3.1%                  |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 4,253                | 2,129                  | 50.1%               | 66.5%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 8,177                | 4,081                  | 49.9%               | 35.6%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 9,044                | 4,498                  | 49.7%               | 28.7%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 10,229               | 4,865                  | 47.6%               | 19.4%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 11,022               | 5,020                  | 45.5%               | 13.2%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 11,422               | 4,987                  | 43.7%               | 10.0%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 11,850               | 4,569                  | 38.6%               | 6.6%                  |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 12,072               | 4,191                  | 34.7%               | 4.9%                  |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 12,296               | 3,704                  | 30.1%               | 3.1%                  |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 8,051                | 3,993                  | 49.6%               | 36.6%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 10,121               | 4,716                  | 46.6%               | 20.3%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 11,367               | 4,830                  | 42.5%               | 10.4%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 12,296               | 3,704                  | 30.1%               | 3.1%                  |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 2,162                | 925                    | 42.8%               | 83.0%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 4,173                | 1,761                  | 42.2%               | 67.1%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 6,512                | 2,727                  | 41.9%               | 48.7%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 7,962                | 3,200                  | 40.2%               | 37.3%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 9,414                | 3,338                  | 35.5%               | 25.8%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 10,226               | 3,271                  | 32.0%               | 19.4%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 11,050               | 3,062                  | 27.7%               | 12.9%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 4,087                | 1,672                  | 40.9%               | 67.8%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 7,798                | 2,976                  | 38.2%               | 38.6%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 11,050               | 3,062                  | 27.7%               | 12.9%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 11,050               | 417                    | 3.8%                | 12.9%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 23,516               | 11,645                 | 49.5%               | 33.4%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 10,897               | 5,498                  | 50.5%               | 30.0%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 8,675                | 4,460                  | 51.4%               | 29.9%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 6,393                | 3,273                  | 51.2%               | 27.8%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 4,526                | 2,299                  | 50.8%               | 26.9%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 3,463                | 1,788                  | 51.6%               | 26.8%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 2,436                | 1,231                  | 50.5%               | 25.7%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 1,846                | 920                    | 49.8%               | 25.1%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 1,246                | 642                    | 51.5%               | 24.1%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 11,192               | 5,734                  | 51.2%               | 30.3%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 6,506                | 3,289                  | 50.6%               | 28.4%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 3,570                | 1,854                  | 51.9%               | 27.1%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 1,246                | 642                    | 51.5%               | 24.1%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 23,516               | 11,645                 | 49.5%               | 33.4%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 10,897               | 5,498                  | 50.5%               | 30.0%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 8,675                | 4,460                  | 51.4%               | 29.9%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 6,393                | 3,273                  | 51.2%               | 27.8%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 4,526                | 2,299                  | 50.8%               | 26.9%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 3,463                | 1,788                  | 51.6%               | 26.8%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 2,436                | 1,231                  | 50.5%               | 25.7%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 1,846                | 920                    | 49.8%               | 25.1%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 1,246                | 642                    | 51.5%               | 24.1%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 11,192               | 5,734                  | 51.2%               | 30.3%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 6,506                | 3,289                  | 50.6%               | 28.4%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 3,570                | 1,854                  | 51.9%               | 27.1%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 1,246                | 642                    | 51.5%               | 24.1%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 13,419               | 6,016                  | 44.8%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 11,538               | 4,995                  | 43.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 9,419                | 4,099                  | 43.5%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 7,036                | 2,964                  | 42.1%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 5,178                | 2,240                  | 43.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 4,164                | 1,754                  | 42.1%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 2,894                | 1,200                  | 41.5%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 2,315                | 923                    | 39.9%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 1,624                | 642                    | 39.5%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 11,887               | 4,835                  | 40.7%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 7,396                | 3,012                  | 40.7%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 4,197                | 1,735                  | 41.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 1,624                | 642                    | 39.5%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 13,419               | 6,016                  | 44.8%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 11,538               | 4,995                  | 43.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 9,419                | 4,099                  | 43.5%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 7,036                | 2,964                  | 42.1%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 5,178                | 2,240                  | 43.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 4,164                | 1,754                  | 42.1%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 2,894                | 1,200                  | 41.5%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 2,315                | 923                    | 39.9%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 1,624                | 642                    | 39.5%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 11,887               | 4,835                  | 40.7%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 7,396                | 3,012                  | 40.7%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 4,197                | 1,735                  | 41.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 1,624                | 642                    | 39.5%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 9,419                | 4,099                  | 43.5%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 7,036                | 2,964                  | 42.1%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 5,178                | 2,240                  | 43.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 4,164                | 1,754                  | 42.1%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 2,894                | 1,200                  | 41.5%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 2,315                | 923                    | 39.9%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 1,624                | 642                    | 39.5%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 4                    | 1                      | 25.0%               | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 7,396                | 3,012                  | 40.7%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 4,197                | 1,735                  | 41.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 1,624                | 642                    | 39.5%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 1,624                | 17                     | 1.0%                | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 1,279                | 502                    | 39.2%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 18,612               | 7,022                  | 37.7%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 16,361               | 6,105                  | 37.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 11,197               | 3,892                  | 34.8%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 8,763                | 3,028                  | 34.6%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 7,258                | 2,590                  | 35.7%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 5,308                | 1,814                  | 34.2%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 4,162                | 1,390                  | 33.4%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 2,948                | 993                    | 33.7%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 18,167               | 6,815                  | 37.5%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 11,398               | 3,981                  | 34.9%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 7,425                | 2,496                  | 33.6%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 2,948                | 993                    | 33.7%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 1,279                | 502                    | 39.2%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 18,612               | 7,022                  | 37.7%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 16,361               | 6,105                  | 37.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 11,197               | 3,892                  | 34.8%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 8,763                | 3,028                  | 34.6%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 7,258                | 2,590                  | 35.7%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 5,308                | 1,814                  | 34.2%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 4,162                | 1,390                  | 33.4%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 2,948                | 993                    | 33.7%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 18,167               | 6,815                  | 37.5%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 11,398               | 3,981                  | 34.9%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 7,425                | 2,496                  | 33.6%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 2,948                | 993                    | 33.7%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 16,361               | 6,105                  | 37.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 11,197               | 3,892                  | 34.8%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 8,763                | 3,028                  | 34.6%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 7,258                | 2,590                  | 35.7%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 5,308                | 1,814                  | 34.2%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 4,162                | 1,390                  | 33.4%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 2,948                | 993                    | 33.7%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 4                    | 2                      | 50.0%               | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 11,398               | 3,981                  | 34.9%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 7,425                | 2,496                  | 33.6%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 2,948                | 993                    | 33.7%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 2,948                | 114                    | 3.9%                | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 4,274                | 1,243                  | 29.1%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 11,847               | 3,533                  | 29.8%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 22,907               | 6,094                  | 26.6%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 29,864               | 6,710                  | 22.5%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 33,846               | 6,310                  | 18.6%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 38,521               | 5,518                  | 14.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 41,058               | 4,636                  | 11.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 43,787               | 3,519                  | 8.0%                | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 3,887                | 1,154                  | 29.7%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 22,125               | 5,961                  | 26.9%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 33,484               | 6,277                  | 18.7%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 43,787               | 3,519                  | 8.0%                | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 4,274                | 1,243                  | 29.1%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 11,847               | 3,533                  | 29.8%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 22,907               | 6,094                  | 26.6%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 29,864               | 6,710                  | 22.5%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 33,846               | 6,310                  | 18.6%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 38,521               | 5,518                  | 14.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 41,058               | 4,636                  | 11.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 43,787               | 3,519                  | 8.0%                | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 3,887                | 1,154                  | 29.7%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 22,125               | 5,961                  | 26.9%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 33,484               | 6,277                  | 18.7%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 43,787               | 3,519                  | 8.0%                | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 11,847               | 3,533                  | 29.8%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 22,907               | 6,094                  | 26.6%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 29,864               | 6,710                  | 22.5%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 33,846               | 6,310                  | 18.6%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 38,521               | 5,518                  | 14.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 41,058               | 4,636                  | 11.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 43,787               | 3,519                  | 8.0%                | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 8                    | 0                      | 0.0%                | 99.8%                 |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 22,125               | 5,961                  | 26.9%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 33,484               | 6,277                  | 18.7%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 43,787               | 3,519                  | 8.0%                | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 43,787               | 1,994                  | 4.6%                | 0.0%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 19,008               | 9,023                  | 47.5%               | 23.7%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 22,542               | 9,230                  | 40.9%               | 9.5%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 23,052               | 8,921                  | 38.7%               | 7.5%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 23,653               | 8,069                  | 34.1%               | 5.1%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 24,067               | 6,921                  | 28.8%               | 3.4%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 24,279               | 6,116                  | 25.2%               | 2.5%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 24,481               | 4,747                  | 19.4%               | 1.7%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 24,597               | 3,871                  | 15.7%               | 1.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 24,704               | 2,842                  | 11.5%               | 0.8%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 22,484               | 9,316                  | 41.4%               | 9.8%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 23,640               | 8,077                  | 34.2%               | 5.1%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 24,281               | 6,198                  | 25.5%               | 2.5%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 24,704               | 2,842                  | 11.5%               | 0.8%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 19,008               | 9,200                  | 48.4%               | 23.7%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 22,542               | 10,007                 | 44.4%               | 9.5%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 23,052               | 9,993                  | 43.3%               | 7.5%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 23,653               | 9,749                  | 41.2%               | 5.1%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 24,067               | 9,346                  | 38.8%               | 3.4%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 24,279               | 9,280                  | 38.2%               | 2.5%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 24,481               | 9,290                  | 37.9%               | 1.7%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 24,597               | 9,384                  | 38.2%               | 1.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 24,704               | 9,922                  | 40.2%               | 0.8%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 22,484               | 10,080                 | 44.8%               | 9.8%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 23,640               | 9,664                  | 40.9%               | 5.1%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 24,281               | 9,283                  | 38.2%               | 2.5%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 24,704               | 9,922                  | 40.2%               | 0.8%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 18,614               | 6,660                  | 35.8%               | 25.3%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 20,498               | 6,451                  | 31.5%               | 17.7%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 21,874               | 5,858                  | 26.8%               | 12.2%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 22,560               | 5,226                  | 23.2%               | 9.4%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 23,316               | 4,190                  | 18.0%               | 6.4%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 23,672               | 3,409                  | 14.4%               | 5.0%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 24,089               | 2,535                  | 10.5%               | 3.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 23                   | 11                     | 47.8%               | 99.9%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 20,393               | 6,371                  | 31.2%               | 18.1%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 22,500               | 5,302                  | 23.6%               | 9.7%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 24,089               | 2,535                  | 10.5%               | 3.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 24,089               | 1,013                  | 4.2%                | 3.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 204                  | 108                    | 52.9%               | 25.5%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 248                  | 107                    | 43.1%               | 9.5%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 253                  | 98                     | 38.7%               | 7.7%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 260                  | 76                     | 29.2%               | 5.1%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 263                  | 77                     | 29.3%               | 4.0%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 267                  | 59                     | 22.1%               | 2.6%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 266                  | 44                     | 16.5%               | 2.9%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 269                  | 37                     | 13.8%               | 1.8%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 269                  | 21                     | 7.8%                | 1.8%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 253                  | 106                    | 41.9%               | 7.7%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 259                  | 97                     | 37.5%               | 5.5%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 264                  | 64                     | 24.2%               | 3.6%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 269                  | 21                     | 7.8%                | 1.8%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 204                  | 111                    | 54.4%               | 25.5%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 248                  | 109                    | 44.0%               | 9.5%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 253                  | 102                    | 40.3%               | 7.7%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 260                  | 91                     | 35.0%               | 5.1%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 263                  | 94                     | 35.7%               | 4.0%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 267                  | 89                     | 33.3%               | 2.6%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 266                  | 73                     | 27.4%               | 2.9%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 269                  | 79                     | 29.4%               | 1.8%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 269                  | 86                     | 32.0%               | 1.8%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 253                  | 109                    | 43.1%               | 7.7%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 259                  | 111                    | 42.9%               | 5.5%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 264                  | 87                     | 33.0%               | 3.6%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 269                  | 86                     | 32.0%               | 1.8%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 213                  | 74                     | 34.7%               | 22.3%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 230                  | 62                     | 27.0%               | 16.1%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 244                  | 70                     | 28.7%               | 10.9%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 252                  | 54                     | 21.4%               | 8.0%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 256                  | 41                     | 16.0%               | 6.6%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 259                  | 34                     | 13.1%               | 5.5%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 263                  | 20                     | 7.6%                | 4.0%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 226                  | 79                     | 35.0%               | 17.5%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 249                  | 59                     | 23.7%               | 9.1%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 263                  | 20                     | 7.6%                | 4.0%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 263                  | 9                      | 3.4%                | 4.0%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 18,926               | 9,304                  | 49.2%               | 23.7%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 22,408               | 9,491                  | 42.4%               | 9.7%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 22,996               | 9,120                  | 39.7%               | 7.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 23,540               | 8,241                  | 35.0%               | 5.1%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 23,976               | 7,167                  | 29.9%               | 3.4%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 24,173               | 6,297                  | 26.0%               | 2.6%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 24,392               | 4,957                  | 20.3%               | 1.7%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 24,507               | 4,034                  | 16.5%               | 1.2%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 24,614               | 2,924                  | 11.9%               | 0.8%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 22,613               | 9,153                  | 40.5%               | 8.9%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 23,698               | 7,922                  | 33.4%               | 4.5%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 24,205               | 5,952                  | 24.6%               | 2.4%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 24,614               | 2,924                  | 11.9%               | 0.8%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 18,926               | 10,583                 | 55.9%               | 23.7%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 22,408               | 12,785                 | 57.1%               | 9.7%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 22,996               | 12,927                 | 56.2%               | 7.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 23,540               | 12,585                 | 53.5%               | 5.1%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 23,976               | 11,909                 | 49.7%               | 3.4%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 24,173               | 11,287                 | 46.7%               | 2.6%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 24,392               | 10,099                 | 41.4%               | 1.7%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 24,507               | 9,158                  | 37.4%               | 1.2%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 24,614               | 7,921                  | 32.2%               | 0.8%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 22,613               | 12,592                 | 55.7%               | 8.9%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 23,698               | 12,273                 | 51.8%               | 4.5%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 24,205               | 10,856                 | 44.9%               | 2.4%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 24,614               | 7,921                  | 32.2%               | 0.8%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 18,539               | 6,783                  | 36.6%               | 25.3%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 20,435               | 6,601                  | 32.3%               | 17.6%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 21,785               | 6,018                  | 27.6%               | 12.2%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 22,466               | 5,412                  | 24.1%               | 9.5%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 23,185               | 4,357                  | 18.8%               | 6.6%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 23,583               | 3,583                  | 15.2%               | 5.0%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 23,981               | 2,592                  | 10.8%               | 3.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 1,968                | 723                    | 36.7%               | 92.1%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 20,680               | 6,294                  | 30.4%               | 16.7%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 22,617               | 5,100                  | 22.5%               | 8.8%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 23,981               | 2,592                  | 10.8%               | 3.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 23,981               | 1,013                  | 4.2%                | 3.3%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 23,816               | 11,685                 | 49.1%               | 12.0%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 26,883               | 11,193                 | 41.6%               | 0.7%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 27,017               | 10,500                 | 38.9%               | 0.2%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 27,064               | 9,032                  | 33.4%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 27,072               | 7,366                  | 27.2%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 27,072               | 6,147                  | 22.7%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 27,072               | 4,381                  | 16.2%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 27,072               | 3,139                  | 11.6%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 27,072               | 1,834                  | 6.8%                | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 26,882               | 10,926                 | 40.6%               | 0.7%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 27,070               | 8,643                  | 31.9%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 27,072               | 5,974                  | 22.1%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 27,072               | 1,834                  | 6.8%                | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 23,816               | 12,586                 | 52.8%               | 12.0%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 26,883               | 13,724                 | 51.1%               | 0.7%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 27,017               | 13,469                 | 49.9%               | 0.2%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 27,064               | 12,675                 | 46.8%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 27,072               | 11,575                 | 42.8%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 27,072               | 11,081                 | 40.9%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 27,072               | 10,209                 | 37.7%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 27,072               | 9,623                  | 35.5%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 27,072               | 9,049                  | 33.4%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 26,882               | 13,512                 | 50.3%               | 0.7%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 27,070               | 12,233                 | 45.2%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 27,072               | 10,750                 | 39.7%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 27,072               | 9,049                  | 33.4%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 25,141               | 9,524                  | 37.9%               | 7.1%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 26,742               | 8,862                  | 33.1%               | 1.2%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 27,052               | 7,357                  | 27.2%               | 0.1%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 27,072               | 6,147                  | 22.7%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 27,072               | 4,381                  | 16.2%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 27,072               | 3,139                  | 11.6%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 27,072               | 1,834                  | 6.8%                | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 1,363                | 531                    | 39.0%               | 95.0%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 26,698               | 8,440                  | 31.6%               | 1.4%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 27,068               | 5,972                  | 22.1%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 27,072               | 1,834                  | 6.8%                | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 27,072               | 1,304                  | 4.8%                | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,014               | 4,501                  | 44.9%               | 1.1%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,124               | 3,490                  | 34.5%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,124               | 3,083                  | 30.5%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,124               | 2,421                  | 23.9%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,124               | 1,819                  | 18.0%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,124               | 1,407                  | 13.9%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,124               | 874                    | 8.6%                | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,124               | 582                    | 5.7%                | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,124               | 283                    | 2.8%                | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,124               | 3,461                  | 34.2%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,124               | 2,458                  | 24.3%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,124               | 1,403                  | 13.9%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 10,124               | 283                    | 2.8%                | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,014               | 4,900                  | 48.9%               | 1.1%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,124               | 4,407                  | 43.5%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,124               | 4,200                  | 41.5%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,124               | 3,779                  | 37.3%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,124               | 3,434                  | 33.9%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,124               | 3,162                  | 31.2%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,124               | 2,959                  | 29.2%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,124               | 2,741                  | 27.1%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,124               | 2,662                  | 26.3%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,124               | 4,397                  | 43.4%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,124               | 3,789                  | 37.4%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,124               | 3,113                  | 30.7%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 10,124               | 2,662                  | 26.3%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 10,122               | 3,081                  | 30.4%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 10,124               | 2,421                  | 23.9%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 10,124               | 1,819                  | 18.0%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 10,124               | 1,407                  | 13.9%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 10,124               | 874                    | 8.6%                | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 10,124               | 582                    | 5.7%                | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 10,124               | 283                    | 2.8%                | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 501                  | 141                    | 28.1%               | 95.1%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 10,124               | 2,458                  | 24.3%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 10,124               | 1,403                  | 13.9%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 10,124               | 283                    | 2.8%                | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 10,124               | 364                    | 3.6%                | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 4,308                | 2,249                  | 52.2%               | 66.4%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 8,191                | 4,145                  | 50.6%               | 36.0%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 9,160                | 4,556                  | 49.7%               | 28.5%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 10,265               | 4,933                  | 48.1%               | 19.8%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 11,110               | 4,980                  | 44.8%               | 13.2%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 11,523               | 4,918                  | 42.7%               | 10.0%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 11,943               | 4,493                  | 37.6%               | 6.7%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 12,177               | 4,221                  | 34.7%               | 4.9%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 12,391               | 3,670                  | 29.6%               | 3.2%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 8,344                | 4,188                  | 50.2%               | 34.8%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 10,403               | 4,995                  | 48.0%               | 18.8%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 11,554               | 4,837                  | 41.9%               | 9.8%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 12,391               | 3,670                  | 29.6%               | 3.2%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 4,308                | 2,408                  | 55.9%               | 66.4%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 8,191                | 4,770                  | 58.2%               | 36.0%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 9,160                | 5,353                  | 58.4%               | 28.5%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 10,265               | 5,971                  | 58.2%               | 19.8%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 11,110               | 6,340                  | 57.1%               | 13.2%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 11,523               | 6,413                  | 55.7%               | 10.0%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 11,943               | 6,294                  | 52.7%               | 6.7%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 12,177               | 6,257                  | 51.4%               | 4.9%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 12,391               | 6,218                  | 50.2%               | 3.2%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 8,344                | 4,872                  | 58.4%               | 34.8%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 10,403               | 6,026                  | 57.9%               | 18.8%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 11,554               | 6,363                  | 55.1%               | 9.8%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 12,391               | 6,218                  | 50.2%               | 3.2%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 2,103                | 912                    | 43.4%               | 83.6%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 4,297                | 1,831                  | 42.6%               | 66.4%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 6,727                | 2,770                  | 41.2%               | 47.5%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 8,082                | 3,138                  | 38.8%               | 36.9%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 9,561                | 3,333                  | 34.9%               | 25.3%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 10,318               | 3,305                  | 32.0%               | 19.4%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 11,137               | 3,030                  | 27.2%               | 13.0%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 127                  | 62                     | 48.8%               | 99.0%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 4,477                | 1,846                  | 41.2%               | 65.0%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 8,174                | 3,086                  | 37.8%               | 36.2%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 11,137               | 3,030                  | 27.2%               | 13.0%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 11,137               | 367                    | 3.3%                | 13.0%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 23,559               | 11,993                 | 50.9%               | 33.4%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 10,824               | 5,536                  | 51.1%               | 30.7%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 8,935                | 4,622                  | 51.7%               | 29.3%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 6,290                | 3,272                  | 52.0%               | 28.8%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 4,403                | 2,219                  | 50.4%               | 27.8%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 3,441                | 1,780                  | 51.7%               | 27.1%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 2,382                | 1,160                  | 48.7%               | 26.5%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 1,859                | 916                    | 49.3%               | 25.2%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 1,254                | 640                    | 51.0%               | 24.8%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 10,855               | 5,531                  | 51.0%               | 30.0%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 6,298                | 3,352                  | 53.2%               | 27.6%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 3,384                | 1,753                  | 51.8%               | 27.0%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 1,254                | 640                    | 51.0%               | 24.8%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 23,559               | 12,922                 | 54.8%               | 33.4%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 10,824               | 6,353                  | 58.7%               | 30.7%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 8,935                | 5,371                  | 60.1%               | 29.3%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 6,290                | 3,827                  | 60.8%               | 28.8%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 4,403                | 2,634                  | 59.8%               | 27.8%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 3,441                | 2,085                  | 60.6%               | 27.1%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 2,382                | 1,385                  | 58.1%               | 26.5%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 1,859                | 1,087                  | 58.5%               | 25.2%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 1,254                | 748                    | 59.6%               | 24.8%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 10,855               | 6,328                  | 58.3%               | 30.0%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 6,298                | 3,821                  | 60.7%               | 27.6%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 3,384                | 2,032                  | 60.0%               | 27.0%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 1,254                | 748                    | 59.6%               | 24.8%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 13,324               | 5,949                  | 44.6%               | 0.2%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 11,542               | 5,024                  | 43.5%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 9,082                | 3,863                  | 42.5%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 7,152                | 3,049                  | 42.6%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 5,404                | 2,306                  | 42.7%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 4,172                | 1,749                  | 41.9%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 2,899                | 1,222                  | 42.2%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 2,243                | 986                    | 44.0%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 1,567                | 627                    | 40.0%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 11,564               | 4,818                  | 41.7%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 7,086                | 2,898                  | 40.9%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 4,186                | 1,701                  | 40.6%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 1,567                | 627                    | 40.0%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 13,324               | 6,434                  | 48.3%               | 0.2%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 11,542               | 6,129                  | 53.1%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 9,082                | 4,864                  | 53.6%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 7,152                | 3,930                  | 54.9%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 5,404                | 3,044                  | 56.3%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 4,172                | 2,356                  | 56.5%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 2,899                | 1,638                  | 56.5%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 2,243                | 1,280                  | 57.1%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 1,567                | 876                    | 55.9%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 11,564               | 5,948                  | 51.4%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 7,086                | 3,756                  | 53.0%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 4,186                | 2,299                  | 54.9%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 1,567                | 876                    | 55.9%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 9,082                | 3,863                  | 42.5%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 7,152                | 3,049                  | 42.6%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 5,404                | 2,306                  | 42.7%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 4,172                | 1,749                  | 41.9%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 2,899                | 1,222                  | 42.2%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 2,243                | 986                    | 44.0%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 1,567                | 627                    | 40.0%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 464                  | 205                    | 44.2%               | 96.0%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 7,086                | 2,898                  | 40.9%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 4,186                | 1,701                  | 40.6%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 1,567                | 627                    | 40.0%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 1,567                | 8                      | 0.5%                | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 1,255                | 493                    | 39.3%               | 0.2%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 18,548               | 7,010                  | 37.8%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 16,554               | 6,107                  | 36.9%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 11,113               | 3,947                  | 35.5%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 8,544                | 2,954                  | 34.6%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 7,261                | 2,542                  | 35.0%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 5,428                | 1,778                  | 32.8%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 4,227                | 1,369                  | 32.4%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 3,005                | 1,026                  | 34.1%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 17,855               | 6,692                  | 37.5%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 11,030               | 3,739                  | 33.9%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 7,110                | 2,389                  | 33.6%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 3,005                | 1,026                  | 34.1%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 1,255                | 538                    | 42.9%               | 0.2%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 18,548               | 8,796                  | 47.4%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 16,554               | 7,934                  | 47.9%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 11,113               | 5,419                  | 48.8%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 8,544                | 4,221                  | 49.4%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 7,261                | 3,712                  | 51.1%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 5,428                | 2,721                  | 50.1%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 4,227                | 2,150                  | 50.9%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 3,005                | 1,580                  | 52.6%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 17,855               | 8,355                  | 46.8%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 11,030               | 5,172                  | 46.9%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 7,110                | 3,502                  | 49.3%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 3,005                | 1,580                  | 52.6%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 16,554               | 6,107                  | 36.9%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 11,113               | 3,947                  | 35.5%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 8,544                | 2,954                  | 34.6%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 7,261                | 2,542                  | 35.0%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 5,428                | 1,778                  | 32.8%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 4,227                | 1,369                  | 32.4%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 3,005                | 1,026                  | 34.1%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 792                  | 309                    | 39.0%               | 95.6%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 11,030               | 3,739                  | 33.9%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 7,110                | 2,389                  | 33.6%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 3,005                | 1,026                  | 34.1%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 3,005                | 99                     | 3.3%                | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 4,284                | 1,258                  | 29.4%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 11,730               | 3,547                  | 30.2%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 22,898               | 6,118                  | 26.7%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 29,955               | 6,686                  | 22.3%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 33,845               | 6,401                  | 18.9%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 38,430               | 5,588                  | 14.5%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 41,044               | 4,671                  | 11.4%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 43,761               | 3,494                  | 8.0%                | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 5,076                | 1,534                  | 30.2%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 23,183               | 6,107                  | 26.3%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 34,070               | 6,371                  | 18.7%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 43,761               | 3,494                  | 8.0%                | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 4,284                | 1,623                  | 37.9%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 11,730               | 4,853                  | 41.4%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 22,898               | 9,249                  | 40.4%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 29,955               | 11,450                 | 38.2%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 33,845               | 12,503                 | 36.9%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 38,430               | 13,718                 | 35.7%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 41,044               | 14,104                 | 34.4%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 43,761               | 14,725                 | 33.6%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 5,076                | 2,150                  | 42.4%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 23,183               | 9,299                  | 40.1%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 34,070               | 12,393                 | 36.4%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 43,761               | 14,725                 | 33.6%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 11,730               | 3,547                  | 30.2%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 22,898               | 6,118                  | 26.7%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 29,955               | 6,686                  | 22.3%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 33,845               | 6,401                  | 18.9%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 38,430               | 5,588                  | 14.5%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 41,044               | 4,671                  | 11.4%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 43,761               | 3,494                  | 8.0%                | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 735                  | 220                    | 29.9%               | 85.5%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 23,183               | 6,107                  | 26.3%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 34,070               | 6,371                  | 18.7%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 43,761               | 3,494                  | 8.0%                | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 43,761               | 1,928                  | 4.4%                | 0.0%                  |

**Table 9 caption.** Apparent classification rates by observed seizure burden and cycle regularity strata. These strata identify where null positives are concentrated.

## Table 10. Assumption-based historical definitions

**Why this table is included.** Historical rules are exploratory operationalizations rather than literal replications, so they are flagged separately. This table keeps them out of the core endpoint table while still showing their null false-positive behavior.

**Code to call.**

In [12]:
historical_rows = summary_tables[
    (summary_tables.table_type == "window_false_positive")
    & (summary_tables.phase_mode == "strict_herzog")
    & (summary_tables.subset == "all")
    & (summary_tables.window_type.isin(["calendar", "full"]))
    & (summary_tables.definition.isin([
        "H1_newmark_penry_any", "H1_newmark_penry_66_7_any",
        "H2_duncan1993_any", "H3_herzog1997_twofold_any",
        "H4_reddy2007_any_phase2x_any"
    ]))
].copy()
historical_rows


,table_type,subset,cohort,window_type,window_value,definition,phase_mode,assumption_based_historical,n_windows,n_classifiable,...,indeterminate_rate,positive_rate_all_attempted,unstable_denominator,interpretation_note,n_participants,pattern_category,indeterminate_reason,n_indeterminate,p_prevalence_ge_39_1,p_prevalence_ge_44_2
949,window_false_positive,all,healthy_ovulatory,calendar,1,H1_newmark_penry_any,strict_herzog,True,50000,38214.0,...,0.23572,0.11588,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
950,window_false_positive,all,healthy_ovulatory,calendar,3,H1_newmark_penry_any,strict_herzog,True,50000,45321.0,...,0.09358,0.08036,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
951,window_false_positive,all,healthy_ovulatory,calendar,4,H1_newmark_penry_any,strict_herzog,True,50000,46302.0,...,0.07396,0.06500,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
952,window_false_positive,all,healthy_ovulatory,calendar,6,H1_newmark_penry_any,strict_herzog,True,50000,47533.0,...,0.04934,0.04644,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
953,window_false_positive,all,healthy_ovulatory,calendar,9,H1_newmark_penry_any,strict_herzog,True,50000,48331.0,...,0.03338,0.03302,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1188,window_false_positive,all,population,calendar,12,H4_reddy2007_any_phase2x_any,strict_herzog,True,50000,48719.0,...,0.02562,0.36298,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1189,window_false_positive,all,population,calendar,18,H4_reddy2007_any_phase2x_any,strict_herzog,True,50000,49139.0,...,0.01722,0.25954,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1190,window_false_positive,all,population,calendar,24,H4_reddy2007_any_phase2x_any,strict_herzog,True,50000,49373.0,...,0.01254,0.20268,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1191,window_false_positive,all,population,calendar,36,H4_reddy2007_any_phase2x_any,strict_herzog,True,50000,49587.0,...,0.00826,0.13574,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN


| Cohort                         | Observation window  | CE definition                        | Classifiable windows | False-positive windows | False-positive rate (95% CI) | Indeterminate windows | Assumption-based historical rule |
| ------------------------------ | ------------------- | ------------------------------------ | -------------------- | ---------------------- | ---------------------------- | --------------------- | -------------------------------- |
| healthy ovulatory              | 3 months            | Newmark-Penry two-thirds sensitivity | 45,321               | 1,874                  | 4.1% (4.0, 4.3)              | 9.4%                  | Yes                              |
| healthy ovulatory              | 3 months            | Newmark-Penry perimenstrual rule     | 45,321               | 4,018                  | 8.9% (8.6, 9.1)              | 9.4%                  | Yes                              |
| healthy ovulatory              | 3 months            | Duncan 1993 ten-day rule             | 45,321               | 3,259                  | 7.2% (7.0, 7.4)              | 9.4%                  | Yes                              |
| healthy ovulatory              | 3 months            | Herzog 1997 twofold rule             | 45,321               | 16,837                 | 37.2% (36.7, 37.6)           | 9.4%                  | Yes                              |
| healthy ovulatory              | 3 months            | Reddy 2007 any-phase twofold rule    | 45,321               | 34,735                 | 76.6% (76.3, 77.0)           | 9.4%                  | Yes                              |
| healthy ovulatory              | 36-month full diary | Newmark-Penry two-thirds sensitivity | 49,605               | 159                    | 0.3% (0.3, 0.4)              | 0.8%                  | Yes                              |
| healthy ovulatory              | 36-month full diary | Newmark-Penry perimenstrual rule     | 49,605               | 424                    | 0.9% (0.8, 0.9)              | 0.8%                  | Yes                              |
| healthy ovulatory              | 36-month full diary | Duncan 1993 ten-day rule             | 49,605               | 302                    | 0.6% (0.5, 0.7)              | 0.8%                  | Yes                              |
| healthy ovulatory              | 36-month full diary | Herzog 1997 twofold rule             | 49,605               | 3,720                  | 7.5% (7.3, 7.7)              | 0.8%                  | Yes                              |
| healthy ovulatory              | 36-month full diary | Reddy 2007 any-phase twofold rule    | 49,605               | 6,643                  | 13.4% (13.1, 13.7)           | 0.8%                  | Yes                              |
| heterogeneous menstruating-age | 3 months            | Newmark-Penry two-thirds sensitivity | 45,198               | 1,675                  | 3.7% (3.5, 3.9)              | 9.6%                  | Yes                              |
| heterogeneous menstruating-age | 3 months            | Newmark-Penry perimenstrual rule     | 45,198               | 3,635                  | 8.0% (7.8, 8.3)              | 9.6%                  | Yes                              |
| heterogeneous menstruating-age | 3 months            | Duncan 1993 ten-day rule             | 45,198               | 2,914                  | 6.4% (6.2, 6.7)              | 9.6%                  | Yes                              |
| heterogeneous menstruating-age | 3 months            | Herzog 1997 twofold rule             | 45,198               | 21,242                 | 47.0% (46.5, 47.5)           | 9.6%                  | Yes                              |
| heterogeneous menstruating-age | 3 months            | Reddy 2007 any-phase twofold rule    | 45,198               | 34,555                 | 76.5% (76.1, 76.8)           | 9.6%                  | Yes                              |
| heterogeneous menstruating-age | 36-month full diary | Newmark-Penry two-thirds sensitivity | 49,587               | 166                    | 0.3% (0.3, 0.4)              | 0.8%                  | Yes                              |
| heterogeneous menstruating-age | 36-month full diary | Newmark-Penry perimenstrual rule     | 49,587               | 375                    | 0.8% (0.7, 0.8)              | 0.8%                  | Yes                              |
| heterogeneous menstruating-age | 36-month full diary | Duncan 1993 ten-day rule             | 49,587               | 326                    | 0.7% (0.6, 0.7)              | 0.8%                  | Yes                              |
| heterogeneous menstruating-age | 36-month full diary | Herzog 1997 twofold rule             | 49,587               | 14,835                 | 29.9% (29.5, 30.3)           | 0.8%                  | Yes                              |
| heterogeneous menstruating-age | 36-month full diary | Reddy 2007 any-phase twofold rule    | 49,587               | 6,787                  | 13.7% (13.4, 14.0)           | 0.8%                  | Yes                              |

**Table 10 caption.** Apparent classification rates for exploratory historical definitions. These rows are deliberately labeled as assumption-based and should not be interpreted as literal historical replications.

## Table 11. Output manifest

**Why this table is included.** The manifest is machine-readable provenance: it lists every analysis artifact, size, checksum, and the assumptions that were not directly derivable from simulator outputs.

**Code to call.**

In [13]:
manifest_files = pd.DataFrame(manifest["files"])
manifest_files.assign(size_mb=manifest_files["bytes"] / 1_000_000)[["path", "size_mb", "sha256"]]


,path,size_mb,sha256
0,outputs/random_start_full_v11_hormone_fix/audi...,7.115461,4776305d50ab78c77c5e58c02dc5800cf95a4c0e286202...
1,outputs/random_start_full_v11_hormone_fix/fig1...,0.021292,d40d7f3027f66036390d5c8ea4a6407a8a3189ef66b67b...
2,outputs/random_start_full_v11_hormone_fix/fig1...,0.134229,514387f629851ed311f2d1187a193ad13536e813f662bf...
3,outputs/random_start_full_v11_hormone_fix/fig1...,0.080102,266bed1bdc84e0aab84439f08205e9ae251023acb2add4...
4,outputs/random_start_full_v11_hormone_fix/fig2...,0.018518,beb97c307b4dfd1edffa670c57c1ab036008bb0353c87c...
5,outputs/random_start_full_v11_hormone_fix/fig2...,0.075687,5fd3bcaa872fb661a423667879250dd12ca431056ed6e2...
6,outputs/random_start_full_v11_hormone_fix/fig2...,0.054041,1570bc0ee4d513d91199bea93b58fe427cfa6a54d2fb01...
7,outputs/random_start_full_v11_hormone_fix/fig3...,0.018280,26fa6bf5c8df6fe5c88502a5874a84638c60606e6e208c...
8,outputs/random_start_full_v11_hormone_fix/fig3...,0.107310,e5223e9b85a61d43f71cd2d0b239075fbb52a1aad870f4...
9,outputs/random_start_full_v11_hormone_fix/fig3...,0.065812,f58ce8ce577a6b6547bc9c9fadea51dbc3198568bdef32...


| Output file                                                                             | Size, MB | SHA-256 prefix      |
| --------------------------------------------------------------------------------------- | -------- | ------------------- |
| outputs/random_start_full_v11_hormone_fix/audit_daily_sample.parquet                    | 7.115    | 4776305d50ab78c7... |
| outputs/random_start_full_v11_hormone_fix/fig1_false_positive_by_window.pdf             | 0.021    | d40d7f3027f66036... |
| outputs/random_start_full_v11_hormone_fix/fig1_false_positive_by_window.png             | 0.134    | 514387f629851ed3... |
| outputs/random_start_full_v11_hormone_fix/fig1_false_positive_by_window.svg             | 0.080    | 266bed1bdc84e0aa... |
| outputs/random_start_full_v11_hormone_fix/fig2_pattern_decomposition.pdf                | 0.019    | beb97c307b4dfd1e... |
| outputs/random_start_full_v11_hormone_fix/fig2_pattern_decomposition.png                | 0.076    | 5fd3bcaa872fb661... |
| outputs/random_start_full_v11_hormone_fix/fig2_pattern_decomposition.svg                | 0.054    | 1570bc0ee4d513d9... |
| outputs/random_start_full_v11_hormone_fix/fig3_study_prevalence_distribution_3month.pdf | 0.018    | 26fa6bf5c8df6fe5... |
| outputs/random_start_full_v11_hormone_fix/fig3_study_prevalence_distribution_3month.png | 0.107    | e5223e9b85a61d43... |
| outputs/random_start_full_v11_hormone_fix/fig3_study_prevalence_distribution_3month.svg | 0.066    | f58ce8ce577a6b65... |
| outputs/random_start_full_v11_hormone_fix/fig4_historical_vs_core_definitions.pdf       | 0.019    | 74e350c07e5ee0a3... |
| outputs/random_start_full_v11_hormone_fix/fig4_historical_vs_core_definitions.png       | 0.077    | e74a30a8109b9caa... |
| outputs/random_start_full_v11_hormone_fix/fig4_historical_vs_core_definitions.svg       | 0.051    | b2acdb16387796c5... |
| outputs/random_start_full_v11_hormone_fix/fig4_indeterminate_vs_fpr_frontier.pdf        | 0.027    | ed11117cd93bfed0... |
| outputs/random_start_full_v11_hormone_fix/fig4_indeterminate_vs_fpr_frontier.png        | 0.155    | 0f6f1bc8757c9f47... |
| outputs/random_start_full_v11_hormone_fix/fig4_indeterminate_vs_fpr_frontier.svg        | 0.099    | 5df8d34bdafe6f3e... |
| outputs/random_start_full_v11_hormone_fix/fig5_qc_null_cycle_day_profile.pdf            | 0.020    | 45c7d0e76b0cca91... |
| outputs/random_start_full_v11_hormone_fix/fig5_qc_null_cycle_day_profile.png            | 0.114    | 2abe84a77b07c4aa... |
| outputs/random_start_full_v11_hormone_fix/fig5_qc_null_cycle_day_profile.svg            | 0.076    | 0e64e3bb3a610118... |
| outputs/random_start_full_v11_hormone_fix/participant_summary.parquet                   | 7.097    | bfd9e8859404d675... |
| outputs/random_start_full_v11_hormone_fix/progress.json                                 | 0.000    | adda31888d79c648... |
| outputs/random_start_full_v11_hormone_fix/study_level_3month.parquet                    | 3.017    | 5853fe50b42a55c0... |
| outputs/random_start_full_v11_hormone_fix/study_level_3month_n30.parquet                | 3.017    | 5853fe50b42a55c0... |
| outputs/random_start_full_v11_hormone_fix/summary_tables.csv                            | 4.110    | 190902c491ef1bb6... |
| outputs/random_start_full_v11_hormone_fix/window_results.parquet                        | 114.773  | 57524c02f6dd3459... |

**Table 11 caption.** Machine-readable output provenance. The checksum prefix is included to support reproducibility checks without making the table unnecessarily wide.

## Publication-ready figures

Each figure is written as PNG for notebook viewing and PDF/SVG for publication workflows. Fractional outcomes are displayed on a 0-100% percentage scale. The code cell below is the function call that regenerates the figure set from the populated output tables.

In [14]:
from paper1_null_ce.core.plots import write_all_figures

# Regenerate PNG and PDF figures from current populated output tables.
write_all_figures(OUTPUT_DIR, summary_tables, study_level, pd.read_parquet(OUTPUT_DIR / "audit_daily_sample.parquet"))


[PosixPath('/Users/dgoldenh/Documents/GitHub/catamenial-epilepsy-sim/outputs/random_start_full_v11_hormone_fix/fig1_false_positive_by_window.png'),
 PosixPath('/Users/dgoldenh/Documents/GitHub/catamenial-epilepsy-sim/outputs/random_start_full_v11_hormone_fix/fig1_false_positive_by_window.pdf'),
 PosixPath('/Users/dgoldenh/Documents/GitHub/catamenial-epilepsy-sim/outputs/random_start_full_v11_hormone_fix/fig1_false_positive_by_window.svg'),
 PosixPath('/Users/dgoldenh/Documents/GitHub/catamenial-epilepsy-sim/outputs/random_start_full_v11_hormone_fix/fig2_pattern_decomposition.png'),
 PosixPath('/Users/dgoldenh/Documents/GitHub/catamenial-epilepsy-sim/outputs/random_start_full_v11_hormone_fix/fig2_pattern_decomposition.pdf'),
 PosixPath('/Users/dgoldenh/Documents/GitHub/catamenial-epilepsy-sim/outputs/random_start_full_v11_hormone_fix/fig2_pattern_decomposition.svg'),
 PosixPath('/Users/dgoldenh/Documents/GitHub/catamenial-epilepsy-sim/outputs/random_start_full_v11_hormone_fix/fig3_study

### Figure 1. False-positive rate by window

Calendar-month false-positive rate for practical monitoring durations, split by cohort.

PDF companion: [PDF version](../outputs/random_start_full_v11_hormone_fix/fig1_false_positive_by_window.pdf)

![Figure 1. False-positive rate by window](../outputs/random_start_full_v11_hormone_fix/fig1_false_positive_by_window.png)

**Figure caption.** Apparent classification rates are shown as percentages among classifiable participant windows for random calendar windows. The dashed reference line marks 5%.

### Figure 2. C-pattern decomposition

Mutually exclusive C1/C2/C3 pattern categories for full-diary windows.

PDF companion: [PDF version](../outputs/random_start_full_v11_hormone_fix/fig2_pattern_decomposition.pdf)

![Figure 2. C-pattern decomposition](../outputs/random_start_full_v11_hormone_fix/fig2_pattern_decomposition.png)

**Figure caption.** Bars show the share of all attempted full-diary windows in each pattern category, including indeterminate windows, under strict Herzog phase labeling.

### Figure 3. Study prevalence distribution

Study-level Monte Carlo distribution for 3-month null studies with n=30, n=50, and n=100.

PDF companion: [PDF version](../outputs/random_start_full_v11_hormone_fix/fig3_study_prevalence_distribution_3month.pdf)

![Figure 3. Study prevalence distribution](../outputs/random_start_full_v11_hormone_fix/fig3_study_prevalence_distribution_3month.png)

**Figure caption.** Each curve summarizes simulated studies using random 3-month windows and the windowed Herzog threshold definition. The y-axis is the proportion of simulated studies; vertical reference lines mark benchmark apparent CE prevalence values.

### Figure 4. Indeterminate versus false-positive frontier

Tradeoff between rejecting underspecified windows and the false-positive rate among classifiable windows.

PDF companion: [PDF version](../outputs/random_start_full_v11_hormone_fix/fig4_indeterminate_vs_fpr_frontier.pdf)

![Figure 4. Indeterminate versus false-positive frontier](../outputs/random_start_full_v11_hormone_fix/fig4_indeterminate_vs_fpr_frontier.png)

**Figure caption.** Each point is a definition-by-window-by-cohort result. Points farther right have more indeterminate windows; points higher on the plot have more false positives among windows that remained classifiable.

### Appendix Figure. Historical versus core definitions

Assumption-based historical rules compared with core protocol definitions.

PDF companion: [PDF version](../outputs/random_start_full_v11_hormone_fix/fig4_historical_vs_core_definitions.pdf)

![Appendix Figure. Historical versus core definitions](../outputs/random_start_full_v11_hormone_fix/fig4_historical_vs_core_definitions.png)

**Figure caption.** The historical definitions are exploratory operationalizations and are plotted next to the core definitions only to show their null false-positive behavior under the same 3-month window setting.

### Appendix QC Figure. Null cycle-day seizure profile

Quality-control cycle-day seizure profile in the daily audit sample.

PDF companion: [PDF version](../outputs/random_start_full_v11_hormone_fix/fig5_qc_null_cycle_day_profile.pdf)

![Appendix QC Figure. Null cycle-day seizure profile](../outputs/random_start_full_v11_hormone_fix/fig5_qc_null_cycle_day_profile.png)

**Figure caption.** The audit sample contains 1% of participant daily rows. Lines show average daily seizure frequency by observed menstrual cycle day with approximate Poisson error bars.

## Interpretation notes

- The notebook is populated from the current files in `outputs/random_start_full_v11_hormone_fix`. If those files were produced by smoke mode, the numerical values are smoke-test values, not the final 100,000-participant estimates.
- Full-study values are produced by running `run_paper1_null_ce.py --config config_random_start_full.yaml --full`, then rebuilding this notebook.
- Exact Herzog 2004 results are intentionally present only for 3-complete-cycle windows.
- Historical definitions are assumption-based operationalizations and should be kept separate from core endpoints.
- The manifest assumptions are part of the analysis record:

  - Definition D uses a participant-full-diary method-of-moments negative-binomial alpha recorded in d_alpha; Poisson robust fallback is recorded in d_reason when statsmodels NB fitting fails. Definition D_window_alpha re-estimates alpha from the analyzed window as a non-oracle sensitivity.
  - HORMONE-CYCLE selected diary day 1 uniformly from the first generated cycle.
  - Healthy ovulatory cohort used hormone_cycler build_patient_profile/render_cycle with ovulation_probability set to 1.0 because simulate_diary does not expose a public force-ovulation knob.
  - Historical definitions H1-H4 are assumption-based operationalizations and are flagged in summary outputs.
  - Study-level Monte Carlo samples each selected participant from a deterministic pool of precomputed random valid 3-month windows to avoid retaining all daily diaries in memory.
  - The hormone simulator exposes medical-factor knobs but no natural prevalence sampler; heterogeneous menstruating-age medical factors were sampled from config.yaml rates.